In [ ]:
!pip install -q langchain langchain-google-genai langchain-community faiss-cpu python-docx pydantic
# =====================================================
# IMPORTS
# =====================================================
import os
import re
import json
import unicodedata
from typing import List

import pandas as pd
from docx import Document
from pydantic import BaseModel, Field

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document as LCDocument

from google.colab import drive, userdata
drive.mount('/content/drive')


In [ ]:
# =====================================================
# CONFIGURAÇÕES
# =====================================================
BASE_PATH = "/content/drive/MyDrive/Scripts/Analise de municipio"

ARQUIVO_PLANILHA = f"{BASE_PATH}/indicadores.xlsx"

COLUNA_MUNICIPIO = "municipio"
COLUNA_POPULACAO = "populacao total estimada do municipio"

# Recomendado: usar variável de ambiente em vez de deixar a chave em texto puro.
# No Colab: Ferramentas > Configurações de secrets, ou os.environ via userdata.
API_KEY = userdata.get("API")
MODELO = "gemini-3.6-flash"

llm = ChatGoogleGenerativeAI(
    model=MODELO,
    google_api_key=API_KEY,
    max_output_tokens=4096,
)

# Modelo com Google Search Grounding — usado em duas etapas controladas:
# 1) contexto externo breve para a Análise Geral;
# 2) validação/atualização pontual de informações em cada dimensão.
# A planilha continua sendo a fonte principal do diagnóstico e dos níveis
# de maturidade; a web não recalcula nem substitui esses indicadores.
llm_pesquisa_web = llm.bind_tools([{"google_search": {}}])

# ---- Embeddings (para classificação semântica de indicadores e RAG sobre os chunks da Carta) ----
MODELO_EMBEDDING = "models/gemini-embedding-001"

embeddings = GoogleGenerativeAIEmbeddings(
    model=MODELO_EMBEDDING,
    google_api_key=API_KEY,
)

# Índices FAISS são persistidos no Drive: gerados uma única vez e reaproveitados
# nas próximas execuções (evita reprocessar embeddings sempre).
FAISS_INDEX_TOPICOS_PATH = f"{BASE_PATH}/faiss_index_topicos"
FAISS_INDEX_CHUNKS_PATH = f"{BASE_PATH}/faiss_index_chunks"

# NOTA: a leitura da Carta original (VersoResumidadaCarta.docx via
# ler_carta_como_base_conceitual) foi substituída pelos chunks temáticos
# pré-extraídos da Carta Brasileira para Cidades Inteligentes (ver célula
# "CHUNKS TEMÁTICOS DA CARTA" abaixo). Não é mais necessário CAMINHO_CARTA.


In [ ]:
# =====================================================
# FUNÇÕES AUXILIARES
# =====================================================
def classificar_porte(populacao):
    if populacao <= 50000:
        return "pequeno porte"
    elif populacao <= 300000:
        return "médio porte"
    else:
        return "grande porte"


def _normalizar(texto):
    """Remove acentos e caixa para facilitar comparação de palavras-chave."""
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = texto.encode("ascii", errors="ignore").decode("utf-8")
    return texto.lower()


In [ ]:
# =====================================================
# CHUNKS TEMÁTICOS DA CARTA (pré-extraídos do PDF oficial)
# =====================================================
# Os chunks abaixo foram extraídos e classificados a partir do texto real
# da "Carta Brasileira para Cidades Inteligentes" (versão oficial em PDF,
# 180 páginas, seção "2.5 Objetivos estratégicos e recomendações").
#
# Processo de extração (feito uma única vez, fora deste notebook):
# 1. Texto extraído com pdftotext -layout, isolando a coluna de corpo de
#    texto e descartando a legenda lateral de atores (GF, GE, GM, ...).
# 2. As 122 recomendações numeradas (ex.: "2.8.1 Sustentabilidade em
#    iluminação pública") foram identificadas e mantidas com redação
#    original, sem cortes no meio de uma recomendação.
# 3. Cada recomendação foi classificada, por palavras-chave de título e
#    corpo, no tópico de maior aderência dentro da estrutura de dimensões
#    abaixo. Recomendações sobre o mesmo tema foram reunidas em um único
#    chunk (sem depender da posição no documento), respeitando ~250–500
#    tokens por chunk e sem sobreposição (overlap) entre chunks.
#
# IMPORTANTE: nem todos os 30 tópicos planejados têm conteúdo dedicado na
# Carta (ela é organizada por 8 objetivos transversais de transformação
# digital, não por setores como saúde, transporte ou saneamento). Os 10
# tópicos abaixo ficaram sem chunk correspondente e são tratados via
# fallback no prompt (ver função gerar_analise_dimensao):
TOPICOS_SEM_CHUNK_NA_CARTA = [
    "economica_transporte",
    "economica_vias_publicas",
    "sociocultural_cultura_e_esporte",
    "sociocultural_saude",
    "sociocultural_seguranca_publica",
    "sociocultural_defesa_civil",
    "meio_ambiente_agua_e_saneamento",
    "meio_ambiente_residuos_solidos",
    "meio_ambiente_areas_verdes",
    "institucional_servicos_publicos_digitais",
]

_CHUNKS_JSON = r"""{"geral_conceito_brasileiro_de_cidades_inteligentes": {"chunk_id": "geral_conceito_brasileiro_de_cidades_inteligentes", "dimensao": "Geral", "topico": "Conceito Brasileiro de Cidades Inteligentes", "texto": "Medidas para o alcance da visão de futuro: Elaborar ou revisar normas, políticas, programas e estratégias para adequá-los à visão de futuro da cidade, conforme estabelecido nos instrumentos de planejamento municipal (exemplos: Plano Diretor PD, Plano Plurianual PPA, Lei de Diretrizes Orçamentárias LDO, Lei Orçamentária Anual LOA). Essa adequação irá garantir que os projetos urbanos, inclusive iniciativas de cidades inteligentes, contribuam para realizar a visão de futuro. [Ver recomendação 1.2.4]\n\nPlanejamento para “cidades inteligentes”: Considerar as determinações do Plano Diretor (ver Estatuto da Cidade) ao elaborar estratégias e planos municipais para a transformação digital. Da mesma forma, considerar as determinações do Plano de Desenvolvimento Urbano Integrado (Estatuto da Metrópole), caso exista. Alinhar o planejamento para “cidades inteligentes” com as recomendações desta Carta e seus desdobramentos em termos de normas, diretrizes e padrões. Exemplos de planos municipais para a transformação digital: Plano Diretor de Cidades Inteligentes e Plano Diretor de Tecnologias de Informação e Comunicação–TICs. 2.5.5. Conectividade digital e integração de equipamentos públicos: Fortalecer iniciativas que integrem instituições e equipamentos públicos de ensino e pesquisa. Para isso, formar parcerias entre instituições de modo a prover redes de infraestrutura digital. Ampliar o modelo de Redes Comunitárias de Ensino e Pesquisa para instituições e equipamentos públicos que atendam outras finalidades.\n\nLaboratórios de experimentação urbana: Incentivar o surgimento de soluções urbanas inovadoras, criando espaços colaborativos transdisciplinares (que possibilitam a cooperação entre diferentes disciplinas e saberes) para cidades inteligentes. Essas ações devem considerar a visão ampla da transformação digital nas cidades. Para garantir que as soluções sejam realizáveis, deve-se focar em pesquisa e experimentação em ambientes reais. Para isso, articular instituições de ensino e pesquisa e outros setores envolvidos na produção de conhecimento, com apoio institucional e jurídico da Administração Pública Municipal. Integrar esses Laboratórios ao Observatório da Transformação Digital nas cidades e a outros fóruns oficiais relacionados à transformação digital [ver recomendação 8.2].", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "geral_diversidade_territorial_e_reducao_de_desigualdades": {"chunk_id": "geral_diversidade_territorial_e_reducao_de_desigualdades", "dimensao": "Geral", "topico": "Diversidade Territorial e Redução de Desigualdades", "texto": "Tipologias urbanas: Estabelecer tipologias (categorias) de território que apoiem a compreensão do urbano no Brasil. Esse trabalho deve ser feito no processo de formulação da Política Nacional de Desenvolvimento Urbano (PNDU). Deve compreender o território a partir de diferentes níveis: municipal, supramunicipal (agrupamento de municípios) e regional. As tipologias também devem se adequar à diversidade territorial do país. O objetivo é orientar agendas, programas e iniciativas para o desenvolvimento urbano sustentável, inclusive de cidades inteligentes, nos três níveis (municipal, supramunicipal e regional). 1.2.2. Instrumentos e metodologias para a diversidade territorial: Desenvolver e adaptar instrumentos e metodologias de informação, planejamento, gestão e governança para o desenvolvimento urbano sustentável, considerando diferentes graus de complexidade. Esses instrumentos e metodologias devem ser adequados às tipologias (categorias de territórios) da Política Nacional de Desenvolvimento (PNDU). Devem considerar a diversidade territorial das cidades brasileiras. Devem ser fáceis de implementar, considerando diferentes capacidades presentes no nível local.\n\nVisão de futuro da cidade: Construir a visão de futuro da cidade de forma participativa e inclusiva. Estabelecer essa visão em instrumentos de planejamento municipal (exemplos: Plano Diretor PD, Plano Plurianual PPA, Lei de Diretrizes Orçamentárias LDO, Lei Orçamentária Anual LOA). Na construção da visão de futuro, considerar a perspectiva e os impactos específicos da transformação digital no território da cidade. Considerar também o contexto regional e as características locais nos aspectos econômico-financeiro, sociocultural, urbano-ambiental e político-institucional. Refletir a visão em metas, com etapas, atividades e prazos associados.\n\nArticulação setorial no território: Desenvolver estratégias para que as políticas, planos e programas de desenvolvimento urbano e de setores afins sejam integradas no território, em todos os níveis de governo. As estratégias devem enfatizar as áreas de urbanismo, habitação, saneamento básico (abastecimento de água potável, esgotamento sanitário, limpeza urbana e manejo de resíduos sólidos, drenagem e manejo das águas pluviais urbanas), mobilidade urbana, segurança hídrica, redução de desastres, meio ambiente e tecnologias de informação e comunicação (TICs).", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "geral_transformacao_digital_adaptada_a_capacidade_municipal": {"chunk_id": "geral_transformacao_digital_adaptada_a_capacidade_municipal", "dimensao": "Geral", "topico": "Transformação Digital Adaptada à Capacidade Municipal", "texto": "Diálogo com órgãos de controle: Estabelecer fóruns regulares de diálogo entre: (1) instituições públicas que formulam e implementam políticas públicas; (2) órgãos de controle dos poderes executivo, legislativo e judiciário; (3) Ministério Público; (4) setores envolvidos; (5) organizações da sociedade civil. Esses fóruns devem ter caráter estratégico na tarefa de construir conjuntamente caminhos e suporte à tomada de decisões sobre a transformação digital nas cidades. O objetivo é assegurar a boa condução das políticas sobre o tema da transformação digital nas cidades, em todos os níveis de governo.\n\nServiços urbanos disruptivos: Estruturar espaços de gestão e governança e usar metodologias ágeis para garantir: (1) a tomada de decisão informada por evidências; e (2) a regulação de soluções urbanas em momento adequado. Exemplos de soluções que demandam essas ações: soluções que usam mecanismos ou tecnologias disruptivas (que causam ruptura com padrões e modelos existentes); soluções que geram bases de dados com informações pessoais ou de interesse público; e soluções que usam ou interferem em espaços públicos urbanos (calçadas, praças, sistema viário, soluções de transporte motorizado ou não motorizado, serviços de entrega) etc. entar o desenvolvimento econômico local no contexto da transformadigital Governo Governo Governo Cooperação Cooperação Federal Estadual Municipal Intragovernamental Intragovernamental Vertical Horizontal Agência Empresas Empresas de Setor Privado Reguladora Concecionárias de Telecomunicações Serviços Públicos Instituições de Instituições Organizações da Ensino Financeiras Sociedade Civil e Pesquisa de Fomento OMENDAÇÕES:\n\nLinhas de pesquisa: Incentivar linhas de pesquisa e bolsas de fomento que favoreçam projetos transdisciplinares. O objetivo é produzir conhecimento científico de ponta e de forma contínua sobre a transformação digital nas cidades e seus impactos. 8.5.2. “Ciberinfraestrutura” para geração de conhecimento sobre desenvolvimento urbano sustentável: Apoiar projetos de pesquisa, desenvolvimento e inovação que precisem de “ciberinfraestrutura” (infraestrutura de sistemas operacionais, gestão e processamento de dados, instrumentos avançados e ambientes de visualização) de grande porte. Para tal apoio, devem-se realizar investimentos de longo prazo e articular iniciativas desse tipo de infraestrutura.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "economica_agua_e_esgoto": {"chunk_id": "economica_agua_e_esgoto", "dimensao": "Econômica", "topico": "Água e Esgoto", "texto": "Estratégias setoriais para transformação digital: Elaborar estratégias setoriais para a transformação digital nas cidades, nas áreas de urbanismo, habitação, saneamento básico (abastecimento de água potável, esgotamento sanitário, limpeza urbana e manejo de resíduos sólidos, drenagem e manejo das águas pluviais urbanas), mobilidade urbana, segurança hídrica, redução de desastres, meio ambiente e tecnologias de informação e comunicação (TICs). As estratégias devem: (1) ser elaboradas com base em metodologia única que permita sua consolidação em uma estratégia global; (2) ser elaboradas de forma alinhada com esta Carta; (3) ser desenvolvidas pelos respectivos setores, com apoio da Comunidade da Carta. Os objetivos são: (a) identificar, organizar e endereçar demandas específicas de cada setor; e (b) permitir uma visão global que evite sobreposições e otimize esforços no território.\n\nGoverno Digital: Formular e implementar estratégias estaduais e municipais de governo digital que sejam adequadas a cada realidade. O objetivo é tornar a Administração Pública mais acessível e mais eficiente ao prover serviços, como indica a Estratégia de Governo Digital e a Estratégia Brasileira para a Transformação Digital. 3.6.1. Ampliar o acesso a serviços públicos e direitos sociais por meio de TICs: Usar tecnologias de informação e comunicação (TICs) para promover o direito à cidade e para ampliar os direitos sociais. Focar em áreas urbanas com carências de serviços públicos e em pessoas e grupos sociais vulneráveis. Para realizar esses direitos, as TICs devem ajudar a simplificar o acesso a serviços de saúde, educação, moradia, transporte, saneamento básico (abastecimento de água potável, esgotamento sanitário, limpeza urbana e manejo de resíduos sólidos, drenagem e manejo das águas pluviais urbanas), telecomunicações (inclusive serviços de internet), lazer e cultura.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "economica_residuos_solidos": {"chunk_id": "economica_residuos_solidos", "dimensao": "Econômica", "topico": "Resíduos Sólidos", "texto": "Logística reversa de produtos eletrônicos: Acelerar e dar transparência à estruturação e à implementação de sistemas de logística reversa (coletar e devolver resíduos sólidos ao setor empresarial ou descartá-los corretamente). Esses sistemas devem incluir, por exemplo, fábricas, importadoras, distribuidoras e comércios de produtos eletroeletrônicos e seus componentes. As empresas devem oferecer às pessoas consumidoras dos itens a possibilidade de devolver os resíduos, sem usar serviços públicos de limpeza urbana ou manejo de resíduos sólidos (Política Nacional de Resíduos Sólidos, Art. 33).", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "economica_conectividade": {"chunk_id": "economica_conectividade", "dimensao": "Econômica", "topico": "Conectividade", "texto": "Planejamento na escala de projetos urbanos: Desenvolver, consolidar e disseminar metodologias para elaborar projetos na escala intermediária da cidade (regiões, conjuntos de bairros ou outro agrupamento de áreas que seja menor que o território municipal). O objetivo é implementar processos de renovação urbana, de estruturação urbana ou de expansão urbana. Usar os projetos como oportunidades para distribuir infraestruturas para inclusão digital no espaço urbano. Na elaboração desses projetos, observar os princípios de desenho universal (que viabiliza o uso por todas as pessoas) e as normas de acessibilidade (Estatuto da Pessoa com Deficiência, Art. 55). 1.5.3. Gestão e governança para o desenvolvimento urbano sustentável: [ver Objetivos Estratégicos 3 e 4]. ernet de qualidade para todas as pessoas overno Governo Cooperação Cooperação stadual Municipal Intragovernamental Intragovernamental Vertical Horizontal esas Empresas de Setor Privado onárias de Telecomunicações s Públicos ituições Organizações da nceiras Sociedade Civil omento à internet: Reconhecer e tornar efetivo o dinet por todas as pessoas (Marco Civil da In4o). Para isso, desenvolver e implantar políticas, infraestrutura. Incluir nessas ações projee suporte para redes de telecomunicações, estação dos serviços de telecomunicações bém outros aspectos relacionados à inclus devem ser feitas respeitando as diretrizes União Federal e Agências Reguladoras. ital para todas as pessoas: Viabilizar a ão da infraestrutura para inclusão digital carecem dessa infraestrutura e em áreconectividade. Manter a infraestrutura arantir a inclusão digital em todas as cinte. Nessas ações, enfatizar os núcleas localidades afastadas. Respeitar as prioridades definidas nas políticas nacionais de desenvolvimento regional, de desenvolvimento urbano e de telecomunicações.\n\nEditais de faixas de frequência: Prever contrapartidas para ampliação da infraestrutura para inclusão digital nos editais de faixas de frequência de serviços de telecomunicações. Priorizar o atendimento de áreas que carecem de infraestrutura de qualidade e o atendimento a todas as cidades e comunidades do país. Os municípios devem acompanhar e viabilizar as implantações decorrentes de leilão de faixas de frequência.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "economica_inovacao": {"chunk_id": "economica_inovacao", "dimensao": "Econômica", "topico": "Inovação", "texto": "Centros de gestão integrada: Implantar centros de informações integradas e protocolos públicos para apoiar a tomada de decisões em tempo real. Priorizar a gestão de emergências e a resposta a desastres. Centros articulados com instituições de Ensino e Pesquisa e com o ecossistema de inovação local. O objetivo dessa articulação é produzir conhecimento e construir respostas para problemas públicos. Para essa finalidade, disponibilizar dados coletados pela infraestrutura digital urbana e de registros administrativos anonimizados. Articular os recursos e meios dos Centros de gestão integrada com os dos laboratórios de experimentação urbana.\n\nConstrução de ambientes para inovação: Promover processos de governança e gestão urbana que sejam interinstitucionais (com cooperação entre diferentes instituições) e colaborativos. O objetivo é construir ambientes político-jurídico-institucionais que sejam: (1) favoráveis à inovação; e (2) adaptados ao contexto territorial e ao nível de atuação das instituições.\n\nPolíticas de inovação: Estimular e integrar fóruns de inovação no setor público que sejam interfederativos (agrupando diferentes entes da federação com interesse compartilhado União, Estados, Municípios e Distrito Federal) e abertos à participação ampla de pessoas, instituições e setores interessados. O objetivo é trocar experiências, construir estratégias, políticas e programas, e formular propostas de aperfeiçoamento legislativo e de mecanismos jurídicos. Essas propostas devem reduzir os obstáculos burocráticos à inovação no setor público, incluindo as relações dos governos com a sociedade e a realização de negócios e contratos com empresas de inovação.\n\nProgramas de fomento à inovação: Promover processos de formação e programas de fomento à inovação e ao desenvolvimento tecnológico. Os objetivos são: (1) orientar ações nos setores público e privado; e (2) apoiar o desenvolvimento urbano e a transformação digital sustentáveis, conforme as necessidades e prioridades locais e regionais. 4.4. Capacidades na administração pública para a transformação digital: Desenvolver capacidades e competências na Administração Pública que sejam voltadas à atuação no contexto da transformação digital e seus desdobramentos territoriais. Implementar e fortalecer programas de desenvolvimento institucional em todos os níveis de governo.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "economica_gestao_urbana": {"chunk_id": "economica_gestao_urbana", "dimensao": "Econômica", "topico": "Gestão Urbana", "texto": "TICs para o diagnóstico e a gestão urbana: Usar ferramentas de geoprocessamento (processamento de dados com localização geográfica) para entender melhor os fenômenos urbanos e para aperfeiçoar a capacidade de gestão dos governos locais. Incorporar nessas ações mecanismos inovadores da ciência de dados. Exemplos: (1) Inteligência Artificial (AI); e (2) análise de grandes quantidades de dados anonimizados (sem elementos que identifiquem as pessoas), conhecidos como Big Data. Respeitar a Lei Geral de Proteção de Dados Pessoais (LGPD). [Ver recomendação 3.2.] 1.5.1.2. Sistema nacional de informações para o desenvolvimento urbano: Identificar, sistematizar e disponibilizar dados e informações públicas que sejam relevantes para o desenvolvimento urbano sustentável. Esses dados e informações devem ser elaborados para formular, implementar e monitorar a Política Nacional de Desenvolvimento Urbano (PNDU). Essas ações têm duas finalidades: (1) apoiar a implementação de iniciativas locais pelos entes federados (União, Estados, Distrito Federal e Municípios) e órgãos interfederativos (que representam mais de um ente federado); e (2) atender ao Art. 16-A do Estatuto da Metrópole. [ver recomendação 3.9.]\n\nPlanejamento urbano interfederativo: Apoiar processos de planejamento urbano integrado e intersetorial (com cooperação entre as diferentes áreas de política pública) nas seguintes realidades: (1) regiões metropolitanas, (2) municípios conurbados (municípios com zonas urbanas unidas) e (3) municípios que apresentem relações de interdependência porque compartilham funções públicas de interesse comum. Esses processos de planejamento devem ser integrados de duas formas: pela elaboração de Planos de Desenvolvimento Urbano Integrado (PDUIs) ou pela elaboração conjunta e simultânea de Planos Diretores municipais (PDs). Ao elaborar os planos, é necessário articular dados, ferramentas, estratégias e as abordagens setoriais que façam parte dos planos municipais específicos.\n\nGestão territorial integrada: Usar sistemas de planejamento integrado e de gestão territorial integrada, com base em plataformas interoperáveis (que trabalham em conjunto para a troca eficaz de informações) de dados georreferenciados (plataformas que possibilitem a troca eficaz de dados com localização geográfica), em todos os níveis de governo. Os sistemas devem ser adequados às diferentes escalas das políticas públicas e respeitar a proteção de dados pessoais. Também devem atender às especificidades, demandas e capacidades locais, nos casos de sistemas municipais.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "economica_servicos_online": {"chunk_id": "economica_servicos_online", "dimensao": "Econômica", "topico": "Serviços Online", "texto": "Identidade digital: Adotar e apoiar a implementação da “identidade digital ao cidadão”, conforme consta da Estratégia de Governo Digital.\n\nCooperação interfederativa em governo digital: Promover o intercâmbio de informações em governo digital. Implementar medidas conjuntas de natureza colaborativa por arranjos de cooperação entre governos. Exemplo: adesão voluntária à Rede Nacional de Governo Digital – Rede Gov.br (Decreto 10.332/20, Art. 7o). O objetivo é otimizar recursos e tempo. 4.2. Atuação em rede e plataformas colaborativas Estado-Sociedade: Mobilizar saberes de diferentes segmentos da sociedade, pessoas e instituições, para construir soluções criativas para problemas urbanos contemporâneos com mais agilidade.\n\nPagamentos digitais de serviços públicos: Facilitar o uso de meios de pagamentos digitais para serviços públicos, desenvolvendo e compartilhando ferramentas que estejam alinhadas com a Plataforma de Cidadania Digital. Adotar o PIX (pagamento instantâneo do Banco Central) como forma de pagamento para serviços públicos. As ações devem ocorrer em todos os níveis de governo e em cooperação interfederativa (entre União, Estados, Municípios e Distrito Federal).\n\nCompetitividade em serviços digitais urbanos: Buscar formas de garantir competitividade aos ecossistemas (conjunto e relações de pessoas e instituições que desenvolvem tecnologia e inovam) de serviços digitais urbanos. Para isso, devem-se usar práticas que evitem monopólios e promovam a escolha livre dos usuários. As ações devem estar alinhadas com a Declaração de Direitos de Liberdade Econômica.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "economica_dados_abertos": {"chunk_id": "economica_dados_abertos", "dimensao": "Econômica", "topico": "Dados Abertos", "texto": "Políticas de dados abertos: Implementar políticas de dados abertos em todos os níveis de governo. Usar experiências e recursos já disponíveis e em operação, tais como: Portal Brasileiro de Dados Abertos, Infraestrutura Nacional de Dados Abertos (INDA) e Infraestrutura Nacional de Dados Espaciais (INDE). Usar as políticas de dados abertos para cumprir o princípio da transparência na administração pública e a Lei de Acesso à Informação (LAI). Usar os modelos e recomendações produzidos pela Parceria para Governo Aberto (OGP Open Government Partnership).\n\nDados geoespaciais: Fortalecer a Infraestrutura Nacional de Dados Espaciais (INDE) como plataforma que facilita o intercâmbio de dados geoespaciais (dados espaciais com localização geográfica). Estabelecer a Política Nacional de Geoinformação (PNGeo) e consolidar um vocabulário uniforme e específico em sistemas de informação geográfica urbana. 3.5.3. Padronização para elaboração de cadastros territoriais: Articular iniciativas governamentais que elaboram, ou contribuem para elaborar, cadastros imobiliários. Essa articulação deve ter como foco uniformizar conceitos, nomenclaturas, métodos e meios de implementação. Isso irá otimizar esforços e garantir a interoperabilidade (capacidade de sistemas trabalharem em conjunto para a troca eficaz de informações) de dados.\n\nRegulação da propriedade de dados: Definir com precisão os direitos sobre a propriedade e as condições para usar dados em contratos públicos e na atuação pública de caráter regulatório. O mesmo deve ocorrer em iniciativas interinstitucionais que impliquem na geração e no compartilhamento de dados, incluindo as iniciativas público-privadas. Priorizar a abertura e uso dos dados em políticas públicas. Em todos os casos mencionados, respeitar o princípio da função social da propriedade, conforme consta do artigo constitucional sobre ordem econômica. (Art. 170 da Constituição Federal).\n\nPlataformas públicas de compartilhamento de dados: Disponibilizar dados abertos e informações públicas em linguagem inclusiva, de forma organizada, compreensível e, sempre que possível, georreferenciados (com localização geográfica). As plataformas de visualização de dados e informações devem ser fáceis de usar por pessoas não-especialistas. Deste modo, as plataformas devem ser programadas em código aberto e com base em softwares livres. Os objetivos são: (1) possibilitar o uso dos dados e das informações pelo ecossistema de inovação local; (2) produzir conhecimento e soluções de interesse público; (3) promover a colaboração para aprimorar dados e análises geradas; e (4) reduzir a dependência de recursos para contratação e manutenção de licenças de softwares.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "sociocultural_educacao": {"chunk_id": "sociocultural_educacao", "dimensao": "Sociocultural", "topico": "Educação", "texto": "Formação e mercado profissional: Estimular a formação profissional na área de TICs (exemplos: programadores, cientistas de dados), por meio de ensino profissionalizante e de nível superior. Fomentar mercado de trabalho para alocação e retenção das pessoas formadas por meio da articulação de estratégias locais que respondam a demandas das cidades, apoiadas pela rede de Institutos Nacionais de Ciência e Tecnologia (INCT).\n\nTransformação digital e educação urbana: Promover ações de comunicação pública inclusiva e acessível que sejam voltadas ao desenvolvimento urbano e à transformação digital sustentáveis. Abordar grandes transformações globais (ex. mudança do clima). O objetivo dessas ações é sensibilizar e ampliar a consciência da sociedade sobre os impactos desses processos.\n\nCidade educadora: Usar a cidade como suporte para a educação urbana. Para isso, deve-se incentivar que as pessoas e instituições deem valor aos recursos naturais, as áreas verdes e espaços públicos, equipamentos e mobiliário urbano. Também deve-se informar o público sobre a história e o significado dos lugares. Essas ações devem ser associadas ao uso de ferramentas de mapeamento colaborativo que levantem e registrem aspectos subjetivos relacionados a espaços urbanos.\n\nLetramento digital nos currículos escolares: Observar, cumprir e ampliar as propostas contidas na Base Nacional Comum Curricular (BNCC) para integrar a cultura digital nos currículos escolares.\n\nCultura digital na comunidade escolar: Estimular processos de capacitação e aprendizagem em tecnologias digitais para toda a comunidade escolar. Desenvolver ações de educação especificas para o letramento digital de pessoas educadoras capacitando-as para atuar como multiplicadoras da inclusão digital. O objetivo é ampliar, agilizar e facilitar o letramento digital desde a infância até a fase adulta.\n\nRecursos digitais na educação formal: Promover o aparelhamento tecnológico das instituições de ensino por meio de laboratórios, equipamentos, programas, ferramentas, softwares e outros recursos digitais.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "sociocultural_inclusao_digital": {"chunk_id": "sociocultural_inclusao_digital", "dimensao": "Sociocultural", "topico": "Inclusão Digital", "texto": "Informações sobre exclusão digital: Entender melhor os fatores associados à exclusão digital. Exemplos: (1) compreender quais são as condições de conectividade dos grupos vulneráveis; e (2) compreender quais são as condições de conexão em cada localização. Para isso, usar dados georreferenciados (com localização geográfica) separados por critérios como renda, raça, gênero, escolaridade e idade. Incluir análises específicas para as pessoas com deficiência. O uso e tratamento dos dados deve respeitar a legislação sobre proteção de dados pessoais (LGPD). 1.2. Visão de território para o desenvolvimento urbano sustentável:\n\nEnfrentamento da exclusão digital: Promover soluções para os diferentes fatores de exclusão digital nas estratégias de universalização e democratização do acesso à internet e a tecnologias digitais. Essas ações devem estar alinhadas com a Estratégia Brasileira de Transformação Digital, para ajudar a alcançar suas metas.\n\nInclusão digital de pessoas com deficiência: Criar e usar soluções, elaborar e difundir normas e procedimentos para ampliar a acessibilidade da pessoa com deficiência à computação e à internet. Realizar essas ações também na oferta de serviços públicos digitais e outras iniciativas de governo digital (Estatuto da Pessoa com Deficiência, Art. 78). Estimular o desenvolvimento de soluções técnicas previstas no Plano Nacional de Internet das Coisas (Decreto 9.854/2019). 2.4.2.Inclusão digital na perspectiva de gênero: Cumprir as metas nacionais para garantir a igualdade de gênero nas seguintes situações: (1) no acesso, nas habilidades de uso e na produção de tecnologias da informação e comunicação; (2) no acesso e na produção do conhecimento científico; e (3) no acesso e na produção de informação, conteúdos de comunicação e mídias (Agenda 2030, ODS 5, 5.b).\n\nLetramento digital: [ver Objetivo Estratégico 7]\n\nOtimização e melhoria de processos administrativos: Estabelecer sistema de processo administrativo eletrônico. Aderir preferencialmente à infraestrutura pública colaborativa do Processo Eletrônico Nacional (PEN) e suas ações, como o Sistema Eletrônico de Informações – SEI. O objetivo é diminuir custos e tornar a tramitação (o andamento) de documentos públicos mais rápida, transparente e acessível. 3.6.3. Serviços analógicos e medidas de transição para o digital: Manter e melhorar procedimentos analógicos e presenciais quando ofertar serviços públicos digitais. Essas ações também devem ser feitas ao implementar medidas de transição, especialmente quando for um serviço essencial. Considerar a grande quantidade de fatores de exclusão digital.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "sociocultural_inclusao_e_equidade": {"chunk_id": "sociocultural_inclusao_e_equidade", "dimensao": "Sociocultural", "topico": "Inclusão e Equidade", "texto": "Integração de dados para a política urbana: Promover a constante integração de setores e instituições para o intercâmbio de dados, como os dados fiscais, de serviços urbanos e de registros imobiliários. Essa integração permitirá entender melhor o uso e a ocupação do solo urbano. Essas ações irão viabilizar a aplicação de instrumentos de política urbana, como o Imposto Predial e Territorial Urbano (IPTU) progressivo no tempo e o Parcelamento, Edificação e Utilização Compulsório (PEUC). 1.5.1.4. Mapeamento de áreas verdes urbanas e serviços ecossistêmicos: Apoiar os municípios e órgãos interfederativos (que representam mais de um ente federado União, Estados, Distrito Federal e Municípios) a mapear as suas áreas verdes urbanas. Essa ação contribuirá com a meta 11.7 do Objetivo de Desenvolvimento Sustentável 11 da Agenda 2030 da ONU. Além das áreas verdes urbanas, apoiar municípios e órgãos interfederativos a mapear, atribuir valor financeiro e gerir de forma responsável seus recursos naturais e serviços ecossistêmicos. Para isso, disponibilizar sistema e metodologia de cadastro que sejam unificados em âmbito nacional.\n\nRede digital para colaboração urbana: Estimular a formação de uma rede para o desenvolvimento urbano sustentável. A rede deve ser multinível (atuar nos níveis nacionais, regionais, estaduais e locais), interinstitucional (cooperação entre diferentes instituições) e intersetorial (com cooperação entre as diferentes áreas de política pública). A rede deve oferecer recursos digitais e inclusivos para realizar trabalhos colaborativos, incluindo a implementação e a retroalimentação desta Carta Brasileira para Cidades Inteligentes. 4.2.2. Rede de assistência técnica remota para ações no território: Expandir e adaptar o modelo da assistência técnica remota baseada em recursos digitais que foi implementado de forma pioneira pela Rede Universitária de Telemedicina. Essa rede de assistência técnica remota deve apoiar órgãos oficiais interfederativos (que agrupam diferentes entes da federação com interesse compartilhado União, Estados, Municípios e Distrito Federal) e municípios para implementar políticas, projetos e ações de desenvolvimento urbano sustentável, incluindo iniciativas de cidades inteligentes. Apoiar principalmente os municípios de menor capacidade institucional.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "sociocultural_participacao_cidada": {"chunk_id": "sociocultural_participacao_cidada", "dimensao": "Sociocultural", "topico": "Participação Cidadã", "texto": "Mapeamentos colaborativos: Ampliar o uso de ferramentas de mapeamento colaborativo na gestão pública como estratégia para mobilizar saberes e engajamento comunitários. Essas ferramentas também são estratégicas no controle social das políticas públicas, especialmente para levantar necessidades habitacionais, bens comuns, ativos urbanos, ambientais e culturais de interesse coletivo. Além disso, contribuem para identificar e gerir conflitos urbanos. Essas ferramentas devem incluir tecnologias assistivas, de forma a possibilitar a participação da pessoa com deficiência ou mobilidade reduzida. Nessas ações, privilegiar o uso de plataformas e ferramentas gratuitas e de código aberto, como o OpenStreetMap. [Ver recomendação 3.9]\n\nGestão democrática das cidades: Estimular o engajamento e a participação pública inclusiva: na elaboração e na revisão do Plano Diretor e de outros instrumentos de planejamento municipal; (1) em aspectos cotidianos de zeladoria e gestão urbana; e (2) na interação governo-pessoas. Esse estímulo deve se dar por meio de mecanismos inovadores e soluções digitais, e com o uso de tecnologias assistivas (com funcionalidade para garantir autonomia, independência, qualidade de vida e inclusão social da pessoa com deficiência ou com mobilidade reduzida). As ações devem estar de acordo com as demandas e necessidades locais e devem ser adequadas às características organizacionais e institucionais do município. Buscar alinhamento com a Estratégia de Governo Digital (Decreto 10.332/2020, objetivo 14.2) e executar a gestão democrática da cidade (Estatuto da Cidade, Capítulo IV).\n\nImpactos locais da transformação digital e controle social: Estimular que os temas do desenvolvimento urbano e da transformação digital sejam discutidos de forma integrada. Para isso, deve-se estimular a articulação institucional de conselhos ou fóruns que debatem sobre esses temas e que atuem no controle social de políticas públicas. Essas instituições devem acompanhar, avaliar e dar suporte à atuação do município sobre os impactos da transformação digital no território. As ações junto aos municípios devem considerar as condições político-institucionais específicas de cada cidade. 8.5. Ciência, tecnologia e inovação para a transformação digital e o desenvolvimento urbano sustentáveis: Mobilizar diferentes setores da sociedade para ampliar a compreensão sobre os impactos da transformação digital nas cidades. Devem ser considerados os impactos sobre os aspectos econômico-financeiro, sociocultural, urbano-ambiental e político-institucional.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "meio_ambiente_qualidade_do_ar_e_emissoes": {"chunk_id": "meio_ambiente_qualidade_do_ar_e_emissoes", "dimensao": "Meio Ambiente", "topico": "Qualidade do Ar e Emissões", "texto": "Transformação digital e meio ambiente: Desenvolver e usar metodologias, dados e indicadores que respondam às mudanças ambientais e climática (aumento da temperatura média global com aumento da ocorrência de eventos climáticos extremos). Atuar nas frentes de adaptação (como prevenção a eventos climáticos extremos – deslizamentos, inundações, secas, erosões etc.) e de mitigação (redução de emissões de carbono).\n\nDecrescimento e economia zero emissões: Incluir perspectivas de decrescimento, descarbonização e outras variáveis inovadoras de sustentabilidade na exploração de novas alternativas de organização social e econômica. Introzudir a redução de desigualdades socioeconomicas e a distribuição de riquezas na discusssão de modelos econômicos verdes, justos e inovadores. O objetivo é lidar com a escassez de recursos naturais e com a precarização do mundo do trabalho.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "meio_ambiente_energia_e_iluminacao_publica": {"chunk_id": "meio_ambiente_energia_e_iluminacao_publica", "dimensao": "Meio Ambiente", "topico": "Energia e Iluminação Pública", "texto": "Eficiência energética e economia circular: Desenvolver projetos, utilizar mecanismos e tecnologias que ampliem a eficiência energética de infraestruturas e edifícios urbanos. Promover processos e desenvolver soluções que incorporem a lógica da economia circular (aproveitamento de resíduos). O objetivo é promover o uso responsável dos recursos naturais e garantir a qualidade de vida das pessoas das atuais e futuras gerações.\n\nProjetos de iluminação pública: Promover a equidade de acesso ao serviço de iluminação pública nas cidades. Nos projetos de expansão e modernização das redes de iluminação pública, priorizar as seguintes áreas: (1) espaços públicos de utilização intensiva; (2) áreas urbanas desservidas; e (3) áreas urbanas inseguras, com índices de violência urbana acima da média da cidade. Essa priorização e as características de cada área devem ser observadas para a definição de padrões luminotécnicos adequados. Implantar projetos de iluminação pública adequados à diversidade dos municípios brasileiros.\n\nSustentabilidade em iluminação pública: Elevar os padrões de eficiência energética em projetos de modernização e expansão da rede de iluminação pública. Nesses projetos, buscar a redução da poluição luminosa (poluição gerada pelo excesso de luz artificial). Promover a gestão eficiente do serviço por meio da adoção de soluções digitais integradas à rede. O objetivo é minimizar impactos da prestação do serviço de iluminação pública no meio ambiente e na saúde humana, assim como melhorar a qualidade de vida das pessoas nas cidades.\n\nAproveitamento da infraestrutura: Considerar a utilização potencial da rede de iluminação pública como infraestrutura de suporte para a oferta de serviços digitais. Buscar esse aproveitamento especialmente nos projetos de modernização e de expansão da rede de iluminação pública. Garantir o compartilhamento em condições justas, razoáveis e não discriminatórias de acesso aos postes de distribuição de energia elétrica. [Ver recomendação 2.6].", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "institucional_governanca_e_planejamento": {"chunk_id": "institucional_governanca_e_planejamento", "dimensao": "Capacidades Institucionais", "topico": "Governança e Planejamento", "texto": "Visão de contexto: Estimular a atuação local com visão de contexto, disponibilizando ferramentas para facilitar que os municípios percebam seus próprios contextos e inserções regionais. O objetivo é qualificar o planejamento e a gestão integrada de suas áreas urbanas, rurais e naturais. Deve haver articulação com outros municípios e demais entes federados (União, Estados, Municípios e Distrito Federal). Essas ações devem estar em linha com a Política Nacional de Desenvolvimento Regional (PNDR) e com a Política Nacional de Desenvolvimento Urbano (PNDU).\n\nIntersetorialidade no planejamento urbano: Construir e consolidar uma visão integrada do planejamento municipal com base nos instrumentos de planejamento setorial. Enfatizar as áreas de urbanismo, habitação, saneamento básico (abastecimento de água potável, esgotamento sanitário, limpeza urbana e manejo de resíduos sólidos, drenagem e manejo das águas pluviais urbanas), mobilidade urbana, segurança hídrica, redução de desastres, meio ambiente e tecnologias de informação e comunicação (TICs). Exemplo de instrumentos de tecnologias de informação e comunicação nas cidades: Plano Diretor de Cidades Inteligentes e Plano Diretor de TICs. O objetivo é possibilitar que as iniciativas sejam implementadas de forma coordenada no território, usando mecanismos locais de gestão e governança. Para isso, devem ser incluídos mecanismos de dados e informações.\n\nGovernança intermunicipal de dados: Estabelecer instituições de cooperação intermunicipal (entre municípios) para implantar, gerir e operar bases de dados, sistemas digitais e soluções compartilhadas de tecnologia de informação e comunicação. O objetivo deve ser otimizar recursos e ampliar a sustentabilidade dessas ações. Exemplos de instituições de cooperação intermunicipal (entre municípios): consórcios públicos, instâncias de governança metropolitana e associações de municípios.\n\nAgências reguladoras: Alinhar normas, técnicas e operações relativas a serviços públicos que requeiram a instalação de infraestruturas no espaço urbano. Para isso, estabelecer espaço de governança permanente entre agências reguladoras desses serviços públicos. Os objetivos são: (1) racionalizar a instalação e a manutenção de infraestruturas no espaço urbano, otimizando sua utilização; (2) assegurar a observância das normas urbanísticas locais pelas concessionárias dos serviços regulados.\n\nValorização de servidores públicos inovadores: Estabelecer mecanismos para identificar servidores públicos inovadores em todos os níveis de governo. Oferecer incentivos e oportunidades para o desenvolvimento e uso das potencialidades dos servidores em trabalhos institucionais e no aprimoramento de políticas públicas. 4.5. Adoção de processos inovadores de gestão e governança no nível local:", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "institucional_infraestrutura_de_ti": {"chunk_id": "institucional_infraestrutura_de_ti", "dimensao": "Capacidades Institucionais", "topico": "Infraestrutura de TI", "texto": "Instrumentos ambientais: Introduzir o conceito e desenvolver projetos de infraestrutura verde em áreas urbanas. Sempre que possível, substituir a infraestrutura cinza pela infraestrutura verde. Integrar as perspectivas de serviços ecossistêmicos e de soluções baseadas na natureza nos instrumentos de política urbana. Estimular o desenvolvimento de regiões produtoras de alimentos próximas dos centros urbanos. Utilizar as TICs para estimular padrões responsáveis de produção e consumo e ativação da economia local.\n\nInteroperabilidade: Garantir a interoperabilidade (capacidade de sistemas trabalharem em conjunto para a troca eficaz de informações) ao implementar soluções de TICs (Tecnologias de Informação e Comunicação) em governos. Garantir a interoperabilidade também em iniciativas interinstitucionais, inclusive público-privadas. Em todos os casos, respeitar e usar normas, padrões e protocolos públicos oficiais (Programa de Interoperabilidade do Governo Eletrônico e-PING).\n\nContratações governamentais de TICs: Instituir, testar e normatizar novos modelos de governos contratarem Tecnologias de Informação e Comunicação (TICs). Essas ações devem ser feitas de forma conjunta, em cooperação intergovernamental (entre governos). Os novos modelos de contratação devem ter como base o uso de softwares livres e códigos abertos. Assegurar a contratação de instituições, entidades e empresas que tenham: (1) compromisso com os direitos humanos; (2) compromisso com a liberdade de expressão; (3) reputação ilibada; (4) comprovada experiência na área; e (5) responsabilidade e compromisso com a coisa pública. Priorizar a contratação de instituições, entidades e empresas locais. Usar mecanismos de colaboração para compartilhar experiências e boas práticas, tal como acontece na Comunidade de TICs da Plataforma GestGov.\n\nPadrões sustentáveis de produção e consumo: Utilizar as TICs para estimular padrões responsáveis de produção e consumo e ativação da economia local.\n\nCrédito para pequenas empresas de TICs: Facilitar o acesso a condições especiais de crédito por pessoas microempreendedoras individuais e por pequenas empresas de TICs (tecnologias de informação e comunicação). Estabelecer incentivos financeiros e técnicos à operação de pequenos provedores de Internet de forma a garantir a provisão e a sustentabilidade de iniciativas de acesso à internet em parceria com o poder público.\n\nTICs para a redução da pobreza urbana: Usar as tecnologias de informação e comunicação para reduzir a pobreza urbana, contribuindo para a Meta 1.4 do Objetivo de Desenvolvimento Sustentável 1.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "institucional_monitoramento_e_transparencia": {"chunk_id": "institucional_monitoramento_e_transparencia", "dimensao": "Capacidades Institucionais", "topico": "Monitoramento e Transparência", "texto": "Dispositivos digitais no ambiente urbano: Estimular o uso de metodologias, dados e indicadores, digitais ou não, para monitorar e avaliar os impactos ambientais causados por infraestruturas e dispositivos digitais nos ambientes urbanos. Promover o uso responsável de recursos nas soluções de modernização tecnológica de serviços urbanos. O objetivo é reduzir a pegada de carbono na transformação digital das cidades.\n\nTransparência nos algoritmos de empresas de TICs: Incentivar que empresas de tecnologia de informação e comunicação digital tenham padrões elevados de transparência sobre os critérios e pressupostos que usam nos seus algoritmos. Possibilitar e fortalecer processos de auditoria algorítmica e fomentar o uso de softwares de código fonte aberto ou livres. Essas ações contribuem e devem estar alinhadas com o Sistema Nacional para a Transformação Digital.\n\nTransparência orçamentária na Administração Pública: Padronizar dados e informações relativos a contas públicas de todos os poderes e níveis de governo. Garantir a qualidade e a interoperabilidade (capacidade de sistemas trabalharem em conjunto para a troca eficaz de informações) desses dados e informações. Incluir mecanismos que permitam a geolocalização de investimentos públicos. Implementar a transparência ativa, adotando portais públicos organizados que facilitem a compreensão e o manuseio dos dados e informações por pessoas não especializadas. Os objetivos são: (1) facilitar o planejamento e a gestão orçamentária, financeira e patrimonial na Administração Pública; (2) permitir a integração de dados e informações; (3) facilitar o controle interno e externo, bem como o controle social das contas públicas. clusivos de governança urbana e fortaleomo gestor de impactos da transformação overno Governo Cooperação Cooperação stadual Municipal Intragovernamental Intragovernamental Vertical Horizontal esas Empresas de Setor Privado onárias de Telecomunicações s Públicos ituições Organizações da nceiras Sociedade Civil omento overnamental: Fortalecer a articulação solidar a governança urbana multinível eis nacional, regional, estadual e local), operação entre diferentes entes da fedeMunicípios e Distrito Federal) e interseentre as diferentes áreas de política púdos governos estaduais e federal no apoio ações e políticas para os contextos os municípios. terministerial: Fortalecer espaço de gocional de âmbito federal para cidades interticipação aberta aos setores interessados. : (1) construir condições para implementar artilhada para cidades inteligentes; e (2) criar condições para a continuidade da plataforma colaborativa da Carta Brasileira para Cidades Inteligentes.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "institucional_dados_e_seguranca_da_informacao": {"chunk_id": "institucional_dados_e_seguranca_da_informacao", "dimensao": "Capacidades Institucionais", "topico": "Dados e Segurança da Informação", "texto": "Apoio técnico e financeiro para a conectividade: Oferecer soluções para implantar e manter infraestrutura para inclusão digital. Isso deve ser feito por meio de apoio técnico e financeiro ou outros mecanismos de prestação de serviços públicos essenciais. Considerar as capacidades governativas dos municípios brasileiros. Considerar também as condições socioeconômicas e a localização da moradia da população beneficiária. Fomentar e facilitar a articulação dos municípios e de entidades supramunicipais (entidades que atuam sobre um agrupamento de municípios) com operadoras de serviços de telecomunicações. ança de dados e de tecnologias, com vacidade overno Governo Cooperação Cooperação stadual Municipal Intragovernamental Intragovernamental Vertical Horizontal esas Empresas de Setor Privado onárias de Telecomunicações s Públicos ituições Organizações da nceiras Sociedade Civil omento ica: Garantir a segurança cibernética ositivos, sistemas, dados e informações iretrizes, normas e procedimentos que idem a confiabilidade de hardwares, sispositivos de acesso pessoal e ferramentivos). dados pessoais: Garantir a proteção de o completamente à Lei Geral de ProteLGPD). Respeitar a titularidade da pesus próprios dados pessoais, garantindo, itos fundamentais de liberdade, intimiegurar que o compartilhamento de dados incípios de finalidade e transparência. ações, estabelecer normas e procedimensenvolvimento seguro e ético de negócios inovadores baseados em dados. Seguir definições estabelecidas pela Agência Nacional de Proteção de Dados (ANPD).\n\nNormas locais de proteção de dados pessoais: Apoiar os municípios para que adéquem normas e procedimentos à Lei Geral de Proteção de Dados Pessoais (LGPD). Nessa ação, regular de forma prioritária:(1) a regulação do tratamento de dados em serviços públicos essenciais; e (2) os cadastros em serviços digitais. Articular ações junto à Autoridade Nacional de Proteção de Dados (ANPD). O objetivo é garantir a coesão entre as políticas de compartilhamento de dados com aplicação geral e as propostas de cidades inteligentes.", "fonte": "Carta Brasileira para Cidades Inteligentes"}}"""

CHUNKS_CARTA = json.loads(_CHUNKS_JSON)  # chunk_id -> {chunk_id, dimensao, topico, texto, fonte}

print(f"🔹 {len(CHUNKS_CARTA)} chunks temáticos carregados da Carta "
      f"(+ {len(TOPICOS_SEM_CHUNK_NA_CARTA)} tópicos sem conteúdo dedicado, tratados via fallback).")


In [ ]:
# =====================================================
# ÍNDICE FAISS DOS CHUNKS DA CARTA (busca semântica / RAG)
# =====================================================
# Permite recuperar os chunks da Carta mais relevantes por similaridade de
# embeddings, em vez de depender apenas do mapeamento fixo tópico -> chunk_id.
# Usado principalmente como fallback para os tópicos listados em
# TOPICOS_SEM_CHUNK_NA_CARTA, que hoje ficam sem nenhum trecho de referência
# (ver selecionar_chunks_dimensao, mais adiante).

def _construir_indice_chunks():
    documentos = [
        LCDocument(
            page_content=f"{c['topico']}: {c['texto']}",
            metadata={
                "chunk_id": c["chunk_id"],
                "dimensao": c["dimensao"],
                "topico": c["topico"],
            },
        )
        for c in CHUNKS_CARTA.values()
    ]
    return FAISS.from_documents(documentos, embeddings)


def carregar_ou_construir_indice_chunks():
    """Carrega o índice FAISS dos chunks do Drive se ele já existir; caso
    contrário, gera os embeddings uma única vez e salva para reuso nas
    próximas execuções (evita custo/tempo de reprocessar sempre)."""
    if os.path.exists(FAISS_INDEX_CHUNKS_PATH):
        return FAISS.load_local(
            FAISS_INDEX_CHUNKS_PATH, embeddings, allow_dangerous_deserialization=True
        )
    indice = _construir_indice_chunks()
    indice.save_local(FAISS_INDEX_CHUNKS_PATH)
    return indice


INDICE_CHUNKS = carregar_ou_construir_indice_chunks()
print(f"🔹 Índice FAISS de chunks pronto ({len(CHUNKS_CARTA)} vetores).")


In [ ]:
# =====================================================
# ESTRUTURA DE DIMENSÕES E TÓPICOS
# =====================================================
CHUNKS_GERAIS = [
    "conceito_brasileiro_de_cidades_inteligentes",
    "diversidade_territorial_e_reducao_de_desigualdades",
    "transformacao_digital_adaptada_a_capacidade_municipal",
]
CHUNKS_ECONOMICA = [
    "agua_e_esgoto", "residuos_solidos", "transporte", "vias_publicas",
    "conectividade", "inovacao", "gestao_urbana", "servicos_online", "dados_abertos",
]
CHUNKS_SOCIOCULTURAL = [
    "educacao", "cultura_e_esporte", "saude", "seguranca_publica",
    "defesa_civil", "inclusao_digital", "inclusao_e_equidade", "participacao_cidada",
]
CHUNKS_MEIO_AMBIENTE = [
    "agua_e_saneamento", "residuos_solidos", "areas_verdes",
    "qualidade_do_ar_e_emissoes", "energia_e_iluminacao_publica",
]
CHUNKS_CAPACIDADES_INSTITUCIONAIS = [
    "governanca_e_planejamento", "infraestrutura_de_ti", "servicos_publicos_digitais",
    "monitoramento_e_transparencia", "dados_e_seguranca_da_informacao",
]

# dimensão -> (prefixo do chunk_id, lista de tópicos)
DIMENSOES = {
    "Econômica": {"prefixo": "economica", "topicos": CHUNKS_ECONOMICA},
    "Sociocultural": {"prefixo": "sociocultural", "topicos": CHUNKS_SOCIOCULTURAL},
    "Meio Ambiente": {"prefixo": "meio_ambiente", "topicos": CHUNKS_MEIO_AMBIENTE},
    "Capacidades Institucionais": {"prefixo": "institucional", "topicos": CHUNKS_CAPACIDADES_INSTITUCIONAIS},
}

# Palavras-chave por tópico — usadas apenas para classificar indicadores
# (não são trechos da Carta, são termos de busca).
PALAVRAS_CHAVE_TOPICO = {
    "agua_e_esgoto": ["água", "esgoto", "saneamento básico", "abastecimento"],
    "residuos_solidos": ["resíduos sólidos", "lixo", "coleta seletiva", "reciclagem"],
    "transporte": ["transporte público", "mobilidade urbana", "ônibus"],
    "vias_publicas": ["vias públicas", "pavimentação", "trânsito", "mobiliário urbano"],
    "conectividade": ["conectividade", "internet", "banda larga", "wi-fi"],
    "inovacao": ["inovação", "empreendedorismo", "startups"],
    "gestao_urbana": ["gestão urbana", "planejamento urbano", "uso do solo", "plano diretor"],
    "servicos_online": ["serviços online", "serviços digitais", "atendimento digital", "governo digital"],
    "dados_abertos": ["dados abertos", "portal de dados", "transparência de dados"],
    "educacao": ["educação", "escola", "ensino"],
    "cultura_e_esporte": ["cultura", "esporte", "lazer"],
    "saude": ["saúde", "atenção primária", "telessaúde", "telemedicina"],
    "seguranca_publica": ["segurança pública", "violência", "policiamento"],
    "defesa_civil": ["defesa civil", "risco", "desastre"],
    "inclusao_digital": ["inclusão digital", "acesso digital", "letramento digital"],
    "inclusao_e_equidade": ["inclusão", "equidade", "acessibilidade", "grupos vulneráveis"],
    "participacao_cidada": ["participação cidadã", "participação social", "controle social"],
    "agua_e_saneamento": ["água", "saneamento", "recursos hídricos"],
    "areas_verdes": ["áreas verdes", "parques", "arborização"],
    "qualidade_do_ar_e_emissoes": ["qualidade do ar", "emissões", "poluição"],
    "energia_e_iluminacao_publica": ["energia", "iluminação pública", "eficiência energética"],
    "governanca_e_planejamento": ["governança", "planejamento estratégico", "plano diretor"],
    "infraestrutura_de_ti": ["infraestrutura de ti", "tecnologia da informação", "sistemas municipais"],
    "servicos_publicos_digitais": ["serviços públicos digitais", "digitalização"],
    "monitoramento_e_transparencia": ["monitoramento", "transparência", "prestação de contas"],
    "dados_e_seguranca_da_informacao": ["proteção de dados", "segurança da informação", "lgpd"],
}


# =====================================================
# MAPA ENTRE INDICADORES, DIMENSÕES E TÓPICOS
# =====================================================
# Construído automaticamente a partir da estrutura real da planilha
# indicadores.xlsx: 85 indicadores setoriais nas 4 dimensões, cada um
# seguido, na própria planilha, por uma coluna "N M Indicador..." com o
# nível de maturidade (0-7) já calculado para aquele indicador.
# MAPA_NIVEL_MATURIDADE guarda essa relação indicador -> coluna de nível,
# usada para priorizar sugestões pelos indicadores de menor maturidade
# (ver estimar_nivel_indicador, na próxima célula).
#
# 3 indicadores do grupo "Habitação" (dimensão Econômica) não têm tópico
# correspondente na taxonomia dos 30 tópicos e por isso NÃO entram no
# mapa (ficam registrados em INDICADORES_NAO_CLASSIFICADOS quando
# encontrados na planilha): "Percentual de domicílios com população
# vivendo em aglomerados subnormais", "Assentamentos urbanos precários"
# e "Programas e ações habitacionais".
_MAPA_INDICADORES_JSON = r"""{"Índice da população total com atendimento de água": {"dimensao": "Econômica", "topico": "Água e Esgoto", "chunk_id": "economica_agua_e_esgoto"}, "Índice da população total com atendimento de esgoto": {"dimensao": "Econômica", "topico": "Água e Esgoto", "chunk_id": "economica_agua_e_esgoto"}, "Índice da população urbana com atendimento de esgoto": {"dimensao": "Econômica", "topico": "Água e Esgoto", "chunk_id": "economica_agua_e_esgoto"}, "Taxa da população coberta com serviço de coleta de resíduos": {"dimensao": "Econômica", "topico": "Resíduos Sólidos", "chunk_id": "economica_residuos_solidos"}, "Coleta seletiva de resíduos no município": {"dimensao": "Econômica", "topico": "Resíduos Sólidos", "chunk_id": "economica_residuos_solidos"}, "Serviços regulares de transporte de passageiros": {"dimensao": "Econômica", "topico": "Transporte", "chunk_id": "economica_transporte"}, "Serviços de compartilhamento de viagens": {"dimensao": "Econômica", "topico": "Transporte", "chunk_id": "economica_transporte"}, "Serviço de informações de transporte público em tempo real": {"dimensao": "Econômica", "topico": "Transporte", "chunk_id": "economica_transporte"}, "Serviços e soluções inteligentes para mobilidade urbana": {"dimensao": "Econômica", "topico": "Transporte", "chunk_id": "economica_transporte"}, "Acessibilidade no transporte público": {"dimensao": "Econômica", "topico": "Transporte", "chunk_id": "economica_transporte"}, "Ciclomobilidade na cidade": {"dimensao": "Econômica", "topico": "Transporte", "chunk_id": "economica_transporte"}, "Índice de pavimentação das vias públicas": {"dimensao": "Econômica", "topico": "Vias Públicas", "chunk_id": "economica_vias_publicas"}, "Escala de acesso a banda larga fixa": {"dimensao": "Econômica", "topico": "Conectividade", "chunk_id": "economica_conectividade"}, "Escala de acesso a banda larga móvel": {"dimensao": "Econômica", "topico": "Conectividade", "chunk_id": "economica_conectividade"}, "Cobertura de acesso a banda larga móvel por tecnologias 3G e 4G": {"dimensao": "Econômica", "topico": "Conectividade", "chunk_id": "economica_conectividade"}, "Cobertura de fibra ótica": {"dimensao": "Econômica", "topico": "Conectividade", "chunk_id": "economica_conectividade"}, "Rede de tecnologia interligando os equipamentos e edifícios públicos": {"dimensao": "Econômica", "topico": "Conectividade", "chunk_id": "economica_conectividade"}, "Escala de acesso a banda larga fixa de alta velocidade": {"dimensao": "Econômica", "topico": "Conectividade", "chunk_id": "economica_conectividade"}, "Números de estações rádio base": {"dimensao": "Econômica", "topico": "Conectividade", "chunk_id": "economica_conectividade"}, "Qualificação profissional e intermediação de mão de obra": {"dimensao": "Econômica", "topico": "Inovação", "chunk_id": "economica_inovacao"}, "Inclusão produtiva urbana": {"dimensao": "Econômica", "topico": "Inovação", "chunk_id": "economica_inovacao"}, "Acesso a crédito, microcrédito e seguro": {"dimensao": "Econômica", "topico": "Inovação", "chunk_id": "economica_inovacao"}, "Geração de trabalho e renda no município": {"dimensao": "Econômica", "topico": "Inovação", "chunk_id": "economica_inovacao"}, "Sistema de informação geográfica da prefeitura": {"dimensao": "Econômica", "topico": "Gestão Urbana", "chunk_id": "economica_gestao_urbana"}, "Centros de comando e controle para gestão da cidade": {"dimensao": "Econômica", "topico": "Gestão Urbana", "chunk_id": "economica_gestao_urbana"}, "Plataforma integrada de cidade inteligente": {"dimensao": "Econômica", "topico": "Gestão Urbana", "chunk_id": "economica_gestao_urbana"}, "Serviços no website da prefeitura": {"dimensao": "Econômica", "topico": "Serviços Online", "chunk_id": "economica_servicos_online"}, "Dados abertos da gestão municipal": {"dimensao": "Econômica", "topico": "Dados Abertos", "chunk_id": "economica_dados_abertos"}, "Indice de equipamentos de tecnologia disponíveis nas escolas públicas municipais": {"dimensao": "Sociocultural", "topico": "Educação", "chunk_id": "sociocultural_educacao"}, "Taxa de analfabetismo": {"dimensao": "Sociocultural", "topico": "Educação", "chunk_id": "sociocultural_educacao"}, "Índice de desenvolvimento da educação básica (IDEB) - anos finais": {"dimensao": "Sociocultural", "topico": "Educação", "chunk_id": "sociocultural_educacao"}, "Vagas no ensino superior": {"dimensao": "Sociocultural", "topico": "Educação", "chunk_id": "sociocultural_educacao"}, "Centros de educação tecnológica": {"dimensao": "Sociocultural", "topico": "Educação", "chunk_id": "sociocultural_educacao"}, "Ações de educação para comunidades específicas": {"dimensao": "Sociocultural", "topico": "Educação", "chunk_id": "sociocultural_educacao"}, "Taxas de distorção idade-série": {"dimensao": "Sociocultural", "topico": "Educação", "chunk_id": "sociocultural_educacao"}, "Percentual de escolas municipais com acesso à internet": {"dimensao": "Sociocultural", "topico": "Educação", "chunk_id": "sociocultural_educacao"}, "Computadores para uso dos alunos": {"dimensao": "Sociocultural", "topico": "Educação", "chunk_id": "sociocultural_educacao"}, "Estrutura de equipamentos culturais e esportivos": {"dimensao": "Sociocultural", "topico": "Cultura e Esporte", "chunk_id": "sociocultural_cultura_e_esporte"}, "Proteção do patrimônio cultural material e imaterial": {"dimensao": "Sociocultural", "topico": "Cultura e Esporte", "chunk_id": "sociocultural_cultura_e_esporte"}, "Serviços on-line para promoção de cultura": {"dimensao": "Sociocultural", "topico": "Cultura e Esporte", "chunk_id": "sociocultural_cultura_e_esporte"}, "Serviços culturais on-line oferecidos para a população": {"dimensao": "Sociocultural", "topico": "Cultura e Esporte", "chunk_id": "sociocultural_cultura_e_esporte"}, "Serviços de telemedicina ou telessaúde": {"dimensao": "Sociocultural", "topico": "Saúde", "chunk_id": "sociocultural_saude"}, "Leitos hospitalares na rede pública municipal": {"dimensao": "Sociocultural", "topico": "Saúde", "chunk_id": "sociocultural_saude"}, "Médicos disponíveis na rede pública municipal": {"dimensao": "Sociocultural", "topico": "Saúde", "chunk_id": "sociocultural_saude"}, "Prontuário eletrônico": {"dimensao": "Sociocultural", "topico": "Saúde", "chunk_id": "sociocultural_saude"}, "Serviços on-line de saúde oferecidos aos pacientes": {"dimensao": "Sociocultural", "topico": "Saúde", "chunk_id": "sociocultural_saude"}, "Índice de risco e proteção à saúde dos nascidos vivos": {"dimensao": "Sociocultural", "topico": "Saúde", "chunk_id": "sociocultural_saude"}, "Mortalidade materna": {"dimensao": "Sociocultural", "topico": "Saúde", "chunk_id": "sociocultural_saude"}, "Soluções em monitoramento para a segurança pública": {"dimensao": "Sociocultural", "topico": "Segurança Pública", "chunk_id": "sociocultural_seguranca_publica"}, "Taxa de homicídios": {"dimensao": "Sociocultural", "topico": "Segurança Pública", "chunk_id": "sociocultural_seguranca_publica"}, "Políticas públicas e ações para segurança pública": {"dimensao": "Sociocultural", "topico": "Segurança Pública", "chunk_id": "sociocultural_seguranca_publica"}, "Soluções de tecnologia para gestão e monitoramento de desastres naturais": {"dimensao": "Sociocultural", "topico": "Defesa Civil", "chunk_id": "sociocultural_defesa_civil"}, "Vulnerabilidade a riscos e desastres naturais": {"dimensao": "Sociocultural", "topico": "Defesa Civil", "chunk_id": "sociocultural_defesa_civil"}, "Promoção de inclusão digital": {"dimensao": "Sociocultural", "topico": "Inclusão Digital", "chunk_id": "sociocultural_inclusao_digital"}, "Cursos de capacitação tecnológica": {"dimensao": "Sociocultural", "topico": "Inclusão Digital", "chunk_id": "sociocultural_inclusao_digital"}, "Políticas públicas para mulheres": {"dimensao": "Sociocultural", "topico": "Inclusão e Equidade", "chunk_id": "sociocultural_inclusao_e_equidade"}, "Inclusão social para grupos específicos": {"dimensao": "Sociocultural", "topico": "Inclusão e Equidade", "chunk_id": "sociocultural_inclusao_e_equidade"}, "Formas presenciais para participação pública": {"dimensao": "Sociocultural", "topico": "Participação Cidadã", "chunk_id": "sociocultural_participacao_cidada"}, "Formas on-line para participação pública": {"dimensao": "Sociocultural", "topico": "Participação Cidadã", "chunk_id": "sociocultural_participacao_cidada"}, "Índice de volume de esgoto coletado": {"dimensao": "Meio Ambiente", "topico": "Água e Saneamento", "chunk_id": "meio_ambiente_agua_e_saneamento"}, "Consumo médio per capita de água": {"dimensao": "Meio Ambiente", "topico": "Água e Saneamento", "chunk_id": "meio_ambiente_agua_e_saneamento"}, "Soluções inteligentes para gestão na distribuição e consumo de água": {"dimensao": "Meio Ambiente", "topico": "Água e Saneamento", "chunk_id": "meio_ambiente_agua_e_saneamento"}, "Índice de perdas na distribuição de água": {"dimensao": "Meio Ambiente", "topico": "Água e Saneamento", "chunk_id": "meio_ambiente_agua_e_saneamento"}, "Índice de volume de esgoto tratado": {"dimensao": "Meio Ambiente", "topico": "Água e Saneamento", "chunk_id": "meio_ambiente_agua_e_saneamento"}, "Percentual de material recolhido pela coleta seletiva": {"dimensao": "Meio Ambiente", "topico": "Resíduos Sólidos", "chunk_id": "meio_ambiente_residuos_solidos"}, "Soluções inteligentes para otimização da coleta de resíduos": {"dimensao": "Meio Ambiente", "topico": "Resíduos Sólidos", "chunk_id": "meio_ambiente_residuos_solidos"}, "Proteção e gestão do meio ambiente e áreas verdes do município": {"dimensao": "Meio Ambiente", "topico": "Áreas Verdes", "chunk_id": "meio_ambiente_areas_verdes"}, "Soluções em monitoramento de gases de efeito estufa e qualidade do ar": {"dimensao": "Meio Ambiente", "topico": "Qualidade do Ar e Emissões", "chunk_id": "meio_ambiente_qualidade_do_ar_e_emissoes"}, "Monitoramento da qualidade do ar": {"dimensao": "Meio Ambiente", "topico": "Qualidade do Ar e Emissões", "chunk_id": "meio_ambiente_qualidade_do_ar_e_emissoes"}, "Soluções inteligentes para gestão do consumo de energia elétrica": {"dimensao": "Meio Ambiente", "topico": "Energia e Iluminação Pública", "chunk_id": "meio_ambiente_energia_e_iluminacao_publica"}, "Soluções para telegestão da iluminação pública": {"dimensao": "Meio Ambiente", "topico": "Energia e Iluminação Pública", "chunk_id": "meio_ambiente_energia_e_iluminacao_publica"}, "Governança Colaborativa - Responsáveis": {"dimensao": "Capacidades Institucionais", "topico": "Governança e Planejamento", "chunk_id": "institucional_governanca_e_planejamento"}, "Incorporação de TICs - Planejamento": {"dimensao": "Capacidades Institucionais", "topico": "Governança e Planejamento", "chunk_id": "institucional_governanca_e_planejamento"}, "Planejamento Estratégico para Transformação Digital": {"dimensao": "Capacidades Institucionais", "topico": "Governança e Planejamento", "chunk_id": "institucional_governanca_e_planejamento"}, "Governança de TI - Práticas": {"dimensao": "Capacidades Institucionais", "topico": "Infraestrutura de TI", "chunk_id": "institucional_infraestrutura_de_ti"}, "Infraestrutura de Hw e Sw - Armazenamento": {"dimensao": "Capacidades Institucionais", "topico": "Infraestrutura de TI", "chunk_id": "institucional_infraestrutura_de_ti"}, "Gestão Integrada de Dados": {"dimensao": "Capacidades Institucionais", "topico": "Serviços Públicos Digitais", "chunk_id": "institucional_servicos_publicos_digitais"}, "Serviços Públicos On-line": {"dimensao": "Capacidades Institucionais", "topico": "Serviços Públicos Digitais", "chunk_id": "institucional_servicos_publicos_digitais"}, "Solicitação de Serviços Públicos": {"dimensao": "Capacidades Institucionais", "topico": "Serviços Públicos Digitais", "chunk_id": "institucional_servicos_publicos_digitais"}, "Segurança de Políticas Públicas - Monitoramento": {"dimensao": "Capacidades Institucionais", "topico": "Monitoramento e Transparência", "chunk_id": "institucional_monitoramento_e_transparencia"}, "Percepção dos Serviços Públicos": {"dimensao": "Capacidades Institucionais", "topico": "Monitoramento e Transparência", "chunk_id": "institucional_monitoramento_e_transparencia"}, "Transparência - Monitoramento": {"dimensao": "Capacidades Institucionais", "topico": "Monitoramento e Transparência", "chunk_id": "institucional_monitoramento_e_transparencia"}, "Transparência - Execução Orçamentária e Financeira": {"dimensao": "Capacidades Institucionais", "topico": "Dados e Segurança da Informação", "chunk_id": "institucional_dados_e_seguranca_da_informacao"}, "Transparência dos Dados - Disponibilização": {"dimensao": "Capacidades Institucionais", "topico": "Dados e Segurança da Informação", "chunk_id": "institucional_dados_e_seguranca_da_informacao"}, "Segurança dos Dados - Práticas": {"dimensao": "Capacidades Institucionais", "topico": "Dados e Segurança da Informação", "chunk_id": "institucional_dados_e_seguranca_da_informacao"}}"""
_MAPA_NIVEL_MATURIDADE_JSON = r"""{"Índice da população total com atendimento de água": "N M Indicador", "Índice da população total com atendimento de esgoto": "N M Indicador.1", "Índice da população urbana com atendimento de esgoto": "N M Indicador.2", "Taxa da população coberta com serviço de coleta de resíduos": "N M Indicador.3", "Coleta seletiva de resíduos no município": "N M Indicador.4", "Percentual de domicílios com população vivendo em aglomerados subnormais": "N M Indicador.5", "Assentamentos urbanos precários ": "N M Indicador.6", "Programas e ações habitacionais": "N M Indicador.7", "Serviços regulares de transporte de passageiros": "N M Indicador.8", "Serviços de compartilhamento de viagens": "N M Indicador.9", "Serviço de informações de transporte público em tempo real": "N M Indicador.10", "Serviços e soluções inteligentes para mobilidade urbana": "N M Indicador.11", "Acessibilidade no transporte público": "N M Indicador.12", "Ciclomobilidade na cidade": "N M Indicador.13", "Índice de pavimentação das vias públicas": "N M Indicador.14", "Escala de acesso a banda larga fixa": "N M Indicador.15", "Escala de acesso a banda larga móvel": "N M Indicador.16", "Cobertura de acesso a banda larga móvel por tecnologias 3G e 4G": "N M Indicador.17", "Cobertura de fibra ótica": "N M Indicador.18", "Rede de tecnologia interligando os equipamentos e edifícios públicos": "N M Indicador.19", "Escala de acesso a banda larga fixa de alta velocidade": "N M Indicador.20", "Números de estações rádio base": "N M Indicador.21", "Qualificação profissional e intermediação de mão de obra": "N M Indicador.22", "Inclusão produtiva urbana": "N M Indicador.23", "Acesso a crédito, microcrédito e seguro": "N M Indicador.24", "Geração de trabalho e renda no município": "N M Indicador.25", "Sistema de informação geográfica da prefeitura": "N M Indicador.26", "Centros de comando e controle para gestão da cidade": "N M Indicador.27", "Plataforma integrada de cidade inteligente": "N M Indicador.28", "Serviços no website da prefeitura": "N M Indicador.29", "Dados abertos da gestão municipal": "N M Indicador.30", "Indice de equipamentos de tecnologia disponíveis nas escolas públicas municipais": "N M Indicador.31", "Taxa de analfabetismo": "N M Indicador.32", "Índice de desenvolvimento da educação básica (IDEB) - anos finais": "N M Indicador.33", "Vagas no ensino superior": "N M Indicador.34", "Centros de educação tecnológica": "N M Indicador.35", "Ações de educação para comunidades específicas": "N M Indicador.36", "Taxas de distorção idade-série": "N M Indicador.37", "Percentual de escolas municipais com acesso à internet": "N M Indicador.38", "Computadores para uso dos alunos": "N M Indicador.39", "Estrutura de equipamentos culturais e esportivos": "N M Indicador.40", "Proteção do patrimônio cultural material e imaterial": "N M Indicador.41", "Serviços on-line para promoção de cultura": "N M Indicador.42", "Serviços culturais on-line oferecidos para a população": "N M Indicador.43", "Serviços de telemedicina ou telessaúde": "N M Indicador.44", "Leitos hospitalares na rede pública municipal": "N M Indicador.45", "Médicos disponíveis na rede pública municipal": "N M Indicador.46", "Prontuário eletrônico": "N M Indicador.47", "Serviços on-line de saúde oferecidos aos pacientes": "N M Indicador.48", "Índice de risco e proteção à saúde dos nascidos vivos": "N M Indicador.49", "Mortalidade materna": "N M Indicador.50", "Soluções em monitoramento para a segurança pública": "N M Indicador.51", "Taxa de homicídios": "N M Indicador.52", "Políticas públicas e ações para segurança pública": "N M Indicador.53", "Soluções de tecnologia para gestão e monitoramento de desastres naturais": "N M Indicador.54", "Vulnerabilidade a riscos e desastres naturais": "N M Indicador.55", "Promoção de inclusão digital": "N M Indicador.56", "Cursos de capacitação tecnológica": "N M Indicador.57", "Políticas públicas para mulheres": "N M Indicador.58", "Inclusão social para grupos específicos": "N M Indicador.59", "Formas presenciais para participação pública": "N M Indicador.60", "Formas on-line para participação pública": "N M Indicador.61", "Índice de volume de esgoto coletado": "N M Indicador.62", "Consumo médio per capita de água": "N M Indicador.63", "Soluções inteligentes para gestão na distribuição e consumo de água": "N M Indicador.64", "Índice de perdas na distribuição de água": "N M Indicador.65", "Índice de volume de esgoto tratado": "N M Indicador.66", "Percentual de material recolhido pela coleta seletiva": "N M Indicador.67", "Soluções inteligentes para otimização da coleta de resíduos": "N M Indicador.68", "Proteção e gestão do meio ambiente e áreas verdes do município": "N M Indicador.69", "Soluções em monitoramento de gases de efeito estufa e qualidade do ar": "N M Indicador.70", "Monitoramento da qualidade do ar": "N M Indicador.71", "Soluções inteligentes para gestão do consumo de energia elétrica": "N M Indicador.72", "Soluções para telegestão da iluminação pública": "N M Indicador.73", "Governança Colaborativa - Responsáveis": "N M Indicador.74", "Incorporação de TICs - Planejamento": "N M Indicador.75", "Planejamento Estratégico para Transformação Digital": "N M Indicador.76", "Governança de TI - Práticas": "N M Indicador.77", "Infraestrutura de Hw e Sw - Armazenamento": "N M Indicador.78", "Gestão Integrada de Dados": "N M Indicador.79", "Serviços Públicos On-line": "N M Indicador.80", "Solicitação de Serviços Públicos": "N M Indicador.81", "Segurança de Políticas Públicas - Monitoramento": "N M Indicador.82", "Percepção dos Serviços Públicos": "N M Indicador.83", "Transparência - Monitoramento": "N M Indicador.84", "Transparência - Execução Orçamentária e Financeira": "N M Indicador.85", "Transparência dos Dados - Disponibilização": "N M Indicador.86", "Segurança dos Dados - Práticas": "N M Indicador.87"}"""

def _normalizar_coluna(texto):
    """Mesma normalização aplicada a df.columns em main() (strip, minúsculas,
    remoção de acentos) — garante que as chaves do mapa batam exatamente
    com os nomes de coluna já normalizados da planilha."""
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", errors="ignore").decode("utf-8")
    return texto


_MAPA_INDICADORES_BRUTO = json.loads(_MAPA_INDICADORES_JSON)
_MAPA_NIVEL_MATURIDADE_BRUTO = json.loads(_MAPA_NIVEL_MATURIDADE_JSON)

MAPA_INDICADORES = {_normalizar_coluna(k): v for k, v in _MAPA_INDICADORES_BRUTO.items()}
MAPA_NIVEL_MATURIDADE = {_normalizar_coluna(k): _normalizar_coluna(v) for k, v in _MAPA_NIVEL_MATURIDADE_BRUTO.items()}

# Colunas que NÃO são indicadores individuais: dados cadastrais/socioeconômicos
# de contexto (PIB, IDH, GINI etc.), colunas "N M Indicador..." (nível de
# maturidade de cada indicador — já usadas via MAPA_NIVEL_MATURIDADE) e
# colunas de agregado por tópico/dimensão (ex.: "Água e esgoto" como score
# somado do tópico, "Econômica" como score da dimensão). Sem essa lista,
# o classificador por palavra-chave acabaria tratando esses agregados como
# se fossem indicadores individuais (ex.: a coluna-agregado "Saúde" seria
# capturada pela mesma palavra-chave do tópico "saude").
_COLUNAS_NAO_INDICADORES_JSON = r"""["Cod. Município", "Município", "Estado", "Avaliada", "População total estimada do município", "PIB per capita do município", "PIB Agropecuária", "PIB Indústria", "PIB Serviços", "PIB Adminstração Pública", "População ocupada com vínculo formal", "Índice de desenvolvimento humano do município (IDH-M)", "Capacidade de pagamento dos municípios (CAPAG)", "Índice de GINI da renda domiciliar per capita", "Empregos em TIC", "Empresas de TICs no municipio", "Número de Campus de Institutos e Universidades Federais", "Número de Empresas em Parques Tecnológicos", "Número de Incubadoras credenciadas - Lei de TIC", "Número de Instituições de Ensino e Pesquisa em PD&I - Lei de TIC", "Número de Centros e/ou Institutos de PD&I - Lei de TIC", "Número de Empresas habilitadas - Lei de TIC", "Número de empresas - Lei do Bem PD&I", "Equipe de TI - Tamanho", "Estrutura Organizacional de TIC", "Incorporação de TICs - Áreas Prioritárias", "Governança Tecnológica - Responsáveis", "Governança de TI - Responsável", "Nível de Maturidade do município", "N M Indicador", "N M Indicador.1", "N M Indicador.2", "N M Indicador.3", "N M Indicador.4", "N M Indicador.5", "N M Indicador.6", "N M Indicador.7", "N M Indicador.8", "N M Indicador.9", "N M Indicador.10", "N M Indicador.11", "N M Indicador.12", "N M Indicador.13", "N M Indicador.14", "N M Indicador.15", "N M Indicador.16", "N M Indicador.17", "N M Indicador.18", "N M Indicador.19", "N M Indicador.20", "N M Indicador.21", "N M Indicador.22", "N M Indicador.23", "N M Indicador.24", "N M Indicador.25", "N M Indicador.26", "N M Indicador.27", "N M Indicador.28", "N M Indicador.29", "N M Indicador.30", "N M Indicador.31", "N M Indicador.32", "N M Indicador.33", "N M Indicador.34", "N M Indicador.35", "N M Indicador.36", "N M Indicador.37", "N M Indicador.38", "N M Indicador.39", "N M Indicador.40", "N M Indicador.41", "N M Indicador.42", "N M Indicador.43", "N M Indicador.44", "N M Indicador.45", "N M Indicador.46", "N M Indicador.47", "N M Indicador.48", "N M Indicador.49", "N M Indicador.50", "N M Indicador.51", "N M Indicador.52", "N M Indicador.53", "N M Indicador.54", "N M Indicador.55", "N M Indicador.56", "N M Indicador.57", "N M Indicador.58", "N M Indicador.59", "N M Indicador.60", "N M Indicador.61", "N M Indicador.62", "N M Indicador.63", "N M Indicador.64", "N M Indicador.65", "N M Indicador.66", "N M Indicador.67", "N M Indicador.68", "N M Indicador.69", "N M Indicador.70", "N M Indicador.71", "N M Indicador.72", "N M Indicador.73", "N M Indicador.74", "N M Indicador.75", "N M Indicador.76", "N M Indicador.77", "N M Indicador.78", "N M Indicador.79", "N M Indicador.80", "N M Indicador.81", "N M Indicador.82", "N M Indicador.83", "N M Indicador.84", "N M Indicador.85", "N M Indicador.86", "N M Indicador.87", "Econômica", "Sociocultural", "Meio Ambiente", "Capacidades Institucionais", "Água e esgoto", "Resíduos sólidos", "Habitação", "Transporte", "Urbanização vias públicas", "Infraestrutura de conectividade", "Inovação", "Sistemas e tecnologia para gestão urbana", "Serviços on-line da prefeitura", "Dados abertos", "Educação", "Cultura", "Saúde", "Segurança Pública", "Gestão de desastres", "Inclusão digital", "Inclusão social", "Participação pública", "Água e esgoto.1", "Resíduos sólidos.1", "Áreas verdes", "Qualidade do ar", "Energia", "Estratégia", "Infraestrutura de Hw e Sw", "Serviços e aplicações", "Monitoramento", "Dados abertos.1"]"""
COLUNAS_NAO_INDICADORES = {_normalizar_coluna(c) for c in json.loads(_COLUNAS_NAO_INDICADORES_JSON)}

# =====================================================
# DADOS CONTEXTUAIS (subconjunto de COLUNAS_NAO_INDICADORES)
# =====================================================
# Dentro de COLUNAS_NAO_INDICADORES existem dois grupos bem distintos:
#   1) metadado puro / agregado da própria metodologia (Cod. Município,
#      Município, Estado, Avaliada, as 88 colunas "N M Indicador...", o
#      "Nível de Maturidade do município" e os scores agregados por
#      tópico/dimensão, ex.: "Água e esgoto", "Educação", "Econômica");
#   2) dado cadastral/socioeconômico e institucional de contexto (PIB,
#      IDH-M, GINI, CAPAG, empregos/empresas de TIC, universidades
#      federais, instituições de PD&I etc.) — esse é o grupo tratado
#      abaixo como DADOS CONTEXTUAIS.
#
# Os dados contextuais:
#   - SÃO usados para complementar a Análise Geral e a análise da
#     dimensão à qual estão associados (ver obter_dados_contextuais);
#   - NÃO são indicadores da metodologia (continuam fora de
#     indicadores_por_dimensao / MAPA_INDICADORES);
#   - NÃO entram no cálculo de nível de maturidade nem na ordenação de
#     prioridades (obter_nivel_indicador / selecionar_chunks_dimensao
#     seguem operando apenas sobre indicadores);
#   - NÃO devem gerar sugestões diretamente — as sugestões continuam
#     fundamentadas exclusivamente nos indicadores de cada dimensão.
#
# O restante de COLUNAS_NAO_INDICADORES (metadado/agregado, grupo 1)
# permanece apenas excluído, como já ocorria antes desta mudança.
#
# Observação sobre ambiguidade: colunas de instituições de pesquisa e
# inovação (PD&I) foram associadas à Dimensão Econômica, por estarem
# alinhadas ao tópico "Inovação" dessa dimensão — em outro contexto
# poderiam igualmente ser lidas como Capacidades Institucionais.
_DADOS_CONTEXTUAIS_JSON = r"""{
    "PIB per capita do município": "Econômica",
    "PIB Agropecuária": "Econômica",
    "PIB Indústria": "Econômica",
    "PIB Serviços": "Econômica",
    "PIB Adminstração Pública": "Econômica",
    "População ocupada com vínculo formal": "Econômica",
    "Capacidade de pagamento dos municípios (CAPAG)": "Econômica",
    "Empregos em TIC": "Econômica",
    "Empresas de TICs no municipio": "Econômica",
    "Número de Empresas em Parques Tecnológicos": "Econômica",
    "Número de Incubadoras credenciadas - Lei de TIC": "Econômica",
    "Número de Instituições de Ensino e Pesquisa em PD&I - Lei de TIC": "Econômica",
    "Número de Centros e/ou Institutos de PD&I - Lei de TIC": "Econômica",
    "Número de Empresas habilitadas - Lei de TIC": "Econômica",
    "Número de empresas - Lei do Bem PD&I": "Econômica",
    "Índice de desenvolvimento humano do município (IDH-M)": "Sociocultural",
    "Índice de GINI da renda domiciliar per capita": "Sociocultural",
    "Número de Campus de Institutos e Universidades Federais": "Sociocultural",
    "Equipe de TI - Tamanho": "Capacidades Institucionais",
    "Estrutura Organizacional de TIC": "Capacidades Institucionais",
    "Incorporação de TICs - Áreas Prioritárias": "Capacidades Institucionais",
    "Governança Tecnológica - Responsáveis": "Capacidades Institucionais",
    "Governança de TI - Responsável": "Capacidades Institucionais"
}"""

_DADOS_CONTEXTUAIS_BRUTO = json.loads(_DADOS_CONTEXTUAIS_JSON)

# chave normalizada (bate com df.columns já normalizado) -> {dimensao, rotulo}
# "rotulo" preserva o nome original, legível, para uso nos prompts.
DADOS_CONTEXTUAIS_MAPA = {
    _normalizar_coluna(coluna): {"dimensao": dimensao, "rotulo": coluna}
    for coluna, dimensao in _DADOS_CONTEXTUAIS_BRUTO.items()
}

INDICADORES_NAO_CLASSIFICADOS = []  # preenchida em tempo de execução


def classificar_indicador(nome_indicador):
    """Retorna {dimensao, topico, chunk_id} para um indicador, ou None.

    Ordem de resolução:
    1) correspondência exata em MAPA_INDICADORES;
    2) fallback por palavra-chave (nome do indicador x PALAVRAS_CHAVE_TOPICO)
       — útil para indicadores novos, ainda não presentes no mapa;
    3) fallback por similaridade semântica de embeddings (nome do indicador
       x tópicos, via INDICE_TOPICOS) — útil quando o indicador usa
       vocabulário diferente do das palavras-chave cadastradas;
    4) se nada for encontrado, registra em INDICADORES_NAO_CLASSIFICADOS.
    """
    if nome_indicador in COLUNAS_NAO_INDICADORES:
        return None  # metadado/agregado conhecido — não é um indicador setorial

    if nome_indicador in MAPA_INDICADORES:
        return MAPA_INDICADORES[nome_indicador]

    nome_norm = _normalizar(nome_indicador)
    melhor, melhor_score = None, 0
    for dimensao, info in DIMENSOES.items():
        for topico in info["topicos"]:
            palavras = PALAVRAS_CHAVE_TOPICO.get(topico, [])
            score = sum(1 for p in palavras if _normalizar(p) in nome_norm)
            if score > melhor_score:
                melhor_score = score
                melhor = {"dimensao": dimensao, "topico": topico.replace("_", " ").title(),
                          "chunk_id": f"{info['prefixo']}_{topico}"}

    if melhor:
        return melhor

    classificacao_embedding = classificar_indicador_por_embedding(nome_indicador)
    if classificacao_embedding:
        return classificacao_embedding

    INDICADORES_NAO_CLASSIFICADOS.append(nome_indicador)
    return None


In [ ]:
# =====================================================
# ÍNDICE FAISS DOS TÓPICOS (fallback semântico de classificação)
# =====================================================
# Usado por classificar_indicador quando um indicador não bate por nome
# exato (MAPA_INDICADORES) nem por palavra-chave (PALAVRAS_CHAVE_TOPICO):
# em vez de descartar o indicador direto para INDICADORES_NAO_CLASSIFICADOS,
# tenta encontrar por similaridade de embeddings o tópico mais próximo.
LIMIAR_RELEVANCIA_TOPICO = 0.75  # 0-1; ajuste empiricamente conforme os resultados


def _construir_indice_topicos():
    documentos = []
    for dimensao, info in DIMENSOES.items():
        for topico in info["topicos"]:
            palavras = PALAVRAS_CHAVE_TOPICO.get(topico, [])
            texto = f"{topico.replace('_', ' ')}: {', '.join(palavras)}"
            documentos.append(
                LCDocument(
                    page_content=texto,
                    metadata={
                        "dimensao": dimensao,
                        "topico": topico.replace("_", " ").title(),
                        "chunk_id": f"{info['prefixo']}_{topico}",
                    },
                )
            )
    return FAISS.from_documents(documentos, embeddings)


def carregar_ou_construir_indice_topicos():
    if os.path.exists(FAISS_INDEX_TOPICOS_PATH):
        return FAISS.load_local(
            FAISS_INDEX_TOPICOS_PATH, embeddings, allow_dangerous_deserialization=True
        )
    indice = _construir_indice_topicos()
    indice.save_local(FAISS_INDEX_TOPICOS_PATH)
    return indice


INDICE_TOPICOS = carregar_ou_construir_indice_topicos()
print("🔹 Índice FAISS de tópicos pronto.")


def classificar_indicador_por_embedding(nome_indicador):
    """Fallback semântico: retorna {dimensao, topico, chunk_id} do tópico
    mais próximo do nome do indicador por similaridade de embeddings, ou
    None se a melhor correspondência ficar abaixo de LIMIAR_RELEVANCIA_TOPICO."""
    resultados = INDICE_TOPICOS.similarity_search_with_relevance_scores(nome_indicador, k=1)
    if not resultados:
        return None
    doc, score = resultados[0]
    if score < LIMIAR_RELEVANCIA_TOPICO:
        return None
    return {
        "dimensao": doc.metadata["dimensao"],
        "topico": doc.metadata["topico"],
        "chunk_id": doc.metadata["chunk_id"],
    }


In [ ]:
# =====================================================
# NÍVEL DE MATURIDADE E SELEÇÃO DE CHUNKS POR DIMENSÃO
# =====================================================
# Nível de maturidade de um indicador, usado apenas para priorizar QUAIS
# tópicos recebem chunks da Carta primeiro (nunca para gerar números no
# texto final — isso é proibido pelo prompt).
#
# Fonte primária: a própria planilha já traz, para cada indicador, uma
# coluna "N M Indicador..." com o nível de maturidade (0-7) calculado
# oficialmente (ver MAPA_NIVEL_MATURIDADE). Só quando um indicador não
# tiver essa coluna pareada (ex.: indicadores novos adicionados depois)
# é que se recorre à estimativa por texto abaixo, como fallback.
NIVEIS_TEXTUAIS = {
    "inexistente": 0, "nao": 0, "não": 0, "ausente": 0,
    "inicial": 1,
    "basico": 2, "básico": 2,
    "em desenvolvimento": 3,
    "em implantacao": 4, "em implantação": 4,
    "intermediario": 5, "intermediário": 5,
    "avancado": 6, "avançado": 6,
    "consolidado": 7, "sim": 7,
}


def estimar_nivel_indicador(valor):
    """Estima um nível de maturidade (0 a 7) a partir do valor bruto do
    indicador (fallback, usado quando não há coluna N M Indicador
    pareada). Retorna None quando não é possível estimar — nesse caso o
    indicador não entra na priorização, mas continua disponível ao modelo."""
    if pd.isna(valor):
        return None
    if isinstance(valor, (int, float)) and not isinstance(valor, bool):
        nivel = float(valor)
        return nivel if 0 <= nivel <= 7 else None
    return NIVEIS_TEXTUAIS.get(_normalizar(valor))


def obter_nivel_indicador(nome_indicador, valor, linha_planilha=None):
    """Retorna o nível de maturidade (0-7) de um indicador, priorizando a
    coluna oficial "N M Indicador..." da planilha (via
    MAPA_NIVEL_MATURIDADE) e caindo para a estimativa por texto quando
    essa coluna não existir ou não puder ser lida."""
    nm_col = MAPA_NIVEL_MATURIDADE.get(nome_indicador)
    if nm_col is not None and linha_planilha is not None and nm_col in linha_planilha.index:
        nivel_real = linha_planilha[nm_col]
        if not pd.isna(nivel_real):
            try:
                nivel_real = float(nivel_real)
                if 0 <= nivel_real <= 7:
                    return nivel_real
            except (TypeError, ValueError):
                pass
    return estimar_nivel_indicador(valor)


def selecionar_chunks_dimensao(dimensao, indicadores_dimensao, linha_planilha=None,
                                usar_fallback_semantico=True):
    """Seleciona os chunks relevantes para uma dimensão:
    - todos os chunks gerais;
    - chunks dos tópicos que têm ao menos um indicador classificado nesta
      dimensão, ordenados dos indicadores de menor maturidade para os de
      maior (prioriza o que precisa de mais atenção);
    - nunca inclui chunks de outra dimensão;
    - tópicos sem chunk direto na Carta (ver TOPICOS_SEM_CHUNK_NA_CARTA):
      se usar_fallback_semantico=True, busca por similaridade de embeddings
      (via INDICE_CHUNKS) o chunk mais próximo dentro da MESMA dimensão;
      se não encontrar nada (ou o fallback estiver desligado), o tópico
      simplesmente fica sem chunk (nunca inventa conteúdo).
    """
    chunks_gerais = [CHUNKS_CARTA[f"geral_{t}"] for t in CHUNKS_GERAIS if f"geral_{t}" in CHUNKS_CARTA]

    pior_nivel_por_chunk = {}
    for nome, valor in indicadores_dimensao.items():
        classificacao = classificar_indicador(nome)
        if not classificacao or classificacao["dimensao"] != dimensao:
            continue
        nivel = obter_nivel_indicador(nome, valor, linha_planilha)
        chunk_id = classificacao["chunk_id"]
        nivel_efetivo = 99 if nivel is None else nivel
        pior_nivel_por_chunk[chunk_id] = min(pior_nivel_por_chunk.get(chunk_id, 99), nivel_efetivo)

    topicos_ordenados = sorted(pior_nivel_por_chunk, key=lambda cid: pior_nivel_por_chunk[cid])

    chunks_topicos = []
    chunk_ids_incluidos = set()
    for cid in topicos_ordenados:
        if cid in CHUNKS_CARTA:
            chunks_topicos.append(CHUNKS_CARTA[cid])
            chunk_ids_incluidos.add(cid)
        elif usar_fallback_semantico and cid in TOPICOS_SEM_CHUNK_NA_CARTA:
            nome_topico = cid.split("_", 1)[1].replace("_", " ")
            resultados = INDICE_CHUNKS.similarity_search(
                nome_topico, k=1, filter={"dimensao": dimensao}, fetch_k=50
            )
            for doc in resultados:
                chunk_id_encontrado = doc.metadata["chunk_id"]
                if chunk_id_encontrado not in chunk_ids_incluidos:
                    chunks_topicos.append(CHUNKS_CARTA[chunk_id_encontrado])
                    chunk_ids_incluidos.add(chunk_id_encontrado)

    return chunks_gerais + chunks_topicos


def obter_dados_contextuais(dimensao, linha_planilha):
    """Retorna {rotulo_original: valor} dos dados contextuais (cadastrais/
    socioeconômicos e institucionais, ex.: PIB, IDH-M, GINI, CAPAG,
    empregos/empresas de TIC, universidades federais, PD&I) associados a
    uma dimensão específica, a partir de DADOS_CONTEXTUAIS_MAPA.

    Uso exclusivo: complementar a Análise Geral e a análise da própria
    dimensão. Nunca usado para calcular nível de maturidade, ordenar
    prioridades, gerar sugestões ou como indicador da metodologia — esses
    papéis continuam restritos aos indicadores classificados via
    classificar_indicador / MAPA_INDICADORES.
    """
    dados = {}
    for chave_normalizada, info in DADOS_CONTEXTUAIS_MAPA.items():
        if info["dimensao"] != dimensao:
            continue
        if chave_normalizada in linha_planilha.index:
            valor = linha_planilha[chave_normalizada]
            if not pd.isna(valor):
                dados[info["rotulo"]] = valor
    return dados


def obter_todos_dados_contextuais(linha_planilha):
    """Retorna todos os dados contextuais disponíveis (de todas as
    dimensões), usados apenas na Análise Geral."""
    dados = {}
    for dimensao in DIMENSOES:
        dados.update(obter_dados_contextuais(dimensao, linha_planilha))
    return dados


In [ ]:
# =====================================================
# MODELOS ESTRUTURADOS DE SAÍDA (validação automática do JSON)
# =====================================================
class AnaliseGeral(BaseModel):
    analise_geral: str = Field(description="Dois parágrafos: pontos positivos e depois desafios/limitações, sem citar números nem o porte do município.")


class AnaliseDimensao(BaseModel):
    analise: str = Field(
        description=(
            "Exatamente dois parágrafos institucionais sobre a dimensão. "
            "O primeiro apresenta a situação atual e os resultados favoráveis; "
            "o segundo interpreta os principais desafios, contrastes e lacunas, "
            "sem repetir o primeiro parágrafo."
        )
    )
    sugestoes: List[str] = Field(
        description=(
            "Exatamente quatro sugestões de melhoria, distintas entre si, "
            "realistas, diretamente relacionadas aos indicadores e compatíveis "
            "com o porte do município."
        )
    )


llm_estruturado_geral = llm.with_structured_output(AnaliseGeral)
llm_estruturado_dimensao = llm.with_structured_output(AnaliseDimensao)


In [ ]:
# =====================================================
# PROMPTS: ANÁLISE GERAL E ANÁLISE POR DIMENSÃO
# =====================================================
INSTRUCAO_COM_BASE = (
    "BASE CONCEITUAL (apenas para orientar o tom institucional; NÃO citar, "
    "NÃO copiar trechos, NÃO mencionar que existe uma base de referência):\n{base_conceitual}"
)
INSTRUCAO_SEM_BASE = (
    "Não há trecho específico da Carta Brasileira para Cidades Inteligentes "
    "disponível para este tópico. Baseie-se exclusivamente nos indicadores "
    "fornecidos e em boas práticas gerais de gestão pública municipal, sem "
    "inventar diretrizes ou citar qualquer documento de referência."
)

TEMPLATE_ANALISE_GERAL = """\
VOCÊ É UM ANALISTA SÊNIOR EM POLÍTICAS PÚBLICAS E PLANEJAMENTO URBANO,
COM EXPERIÊNCIA EM DESENVOLVIMENTO URBANO E MODERNIZAÇÃO DA GESTÃO MUNICIPAL
NO CONTEXTO BRASILEIRO.

{instrucao_base}

MUNICÍPIO: {municipio}
PORTE POPULACIONAL: {porte}

DADOS CONTEXTUAIS DO MUNICÍPIO
{dados_contextuais_txt}

CONTEXTO EXTERNO — PESQUISA WEB BREVE
{pesquisa_web_txt}

INDICADORES DISPONÍVEIS
{indicadores_txt}


ESTRUTURA OBRIGATÓRIA — SIGA RIGOROSAMENTE:


Considerar o porte do município em todos os textos, com análises mais simples
para municípios menores. Não sugerir soluções complexas incompatíveis com
municípios de pequeno porte. Não usar o termo incipiente.
- Não listar múltiplos indicadores ou evidências no mesmo parágrafo
- No máximo UM exemplo concreto por parágrafo, usado apenas para sustentar a análise
- Priorizar interpretação institucional, evitando enumeração de dados
- Os INDICADORES e os DADOS CONTEXTUAIS DA PLANILHA são a fonte principal do diagnóstico.
- A PESQUISA WEB é apenas complementar: use-a para contextualizar características
  reais do município que ajudem a explicar seu perfil territorial, econômico,
  acadêmico, institucional ou tecnológico.
- Não use a pesquisa web para criar ou alterar níveis de maturidade.
- Se a pesquisa web divergir da planilha, prevalecem os dados da planilha.
- Não transforme fatos externos em relações de causa e efeito sem evidência.
- Use no máximo UM fato proveniente da pesquisa web em cada parágrafo.
- Ignore fatos promocionais, rankings, opiniões ou notícias isoladas que não sejam
  úteis para compreender estruturalmente o município.


→ Dois parágrafos de análise institucional geral, linguagem simples, não citar o porte do município
   - Primeiro parágrafo: pontos positivos
   - Segundo parágrafo: desafios e limitações

REGRAS ABSOLUTAS:
- NÃO encerrar o texto antes de concluir TODAS as partes
- NÃO usar números, índices ou valores
- NÃO mencionar inteligência artificial
- NÃO mencionar a base conceitual ou documentos de referência
- Linguagem acessível, institucional e clara
- Evitar linguagem excessivamente técnica ou abstrata
- Priorizar frases mais diretas e naturais
- Escrever como um diagnóstico institucional real,
  e não como texto acadêmico ou promocional
  Regras obrigatórias:

Utilize linguagem técnica, mas simples e direta.
Priorize frases de tamanho curto ou médio.
Descreva os resultados encontrados antes de fazer interpretações.
Evite elogios excessivos ao município.
Evite transformar todo resultado positivo em uma "potencialidade" ou todo resultado negativo em uma "oportunidade".
Não utilize linguagem de consultoria, marketing ou textos promocionais.
Evite conclusões genéricas que não estejam diretamente relacionadas aos indicadores analisados.
Não exagere a importância dos resultados.
Quando houver aspectos positivos e negativos, apresente-os de maneira equilibrada e factual.
As recomendações devem ser práticas e compatíveis com os problemas identificados.

EVITE EXPRESSÕES TÍPICAS DE TEXTOS GERADOS POR IA, como:

"pilares robustos"
"ambiente fértil"
"potencial significativo"
"caminho promissor"
"desafios significativos"
"oportunidades estratégicas"
"ativo valioso"
"efervescência"
"impulsionar a inovação"
"elevar a eficiência"
"fortalecer a visão de futuro"
"representa uma oportunidade"
"demonstra sólido compromisso"
"se destaca por"
"é fundamental para"
"desempenha papel fundamental"


Varie a construção das frases de forma natural.

PREFIRA formulações mais concretas.

Em vez de:
"O município demonstra um sólido compromisso com a transformação digital."

Escreva:
"O município já oferece parte dos serviços públicos pela internet e utiliza sistemas digitais em algumas áreas da administração."

Em vez de:
"Esse cenário representa uma oportunidade estratégica para ampliar a inovação."

Escreva:
"Esses serviços ainda podem ser ampliados e integrados."

Em vez de:
"A presença das universidades cria um ambiente fértil para a inovação."

Escreva:
"A presença das universidades facilita a aproximação entre a prefeitura, pesquisadores e empresas locais."

Não invente benefícios, causas ou relações. Quando utilizar um fato da pesquisa web,
ele deve estar explicitamente presente no contexto externo fornecido.
"""

TEMPLATE_ANALISE_DIMENSAO = """\
VOCÊ É UM ANALISTA SÊNIOR EM POLÍTICAS PÚBLICAS, TRANSFORMAÇÃO DIGITAL
E PLANEJAMENTO URBANO MUNICIPAL NO CONTEXTO BRASILEIRO.

{instrucao_base}

MUNICÍPIO: {municipio}
PORTE POPULACIONAL: {porte}
DIMENSÃO ANALISADA: {dimensao}

INDICADORES DA DIMENSÃO:
{indicadores_txt}

DADOS CONTEXTUAIS DA DIMENSÃO
{dados_contextuais_txt}

PESQUISA WEB DE VALIDAÇÃO E ATUALIZAÇÃO DA DIMENSÃO
{pesquisa_web_dimensao_txt}

TAREFA:
Produza EXATAMENTE DOIS PARÁGRAFOS de análise institucional sobre a dimensão
{dimensao}, seguidos de EXATAMENTE QUATRO sugestões de melhoria.

ESTRUTURA OBRIGATÓRIA DA ANÁLISE:
- Primeiro parágrafo: descreva a situação atual e os principais resultados
  favoráveis efetivamente sustentados pelos indicadores. Relacione evidências
  complementares quando isso ajudar a formar uma leitura coerente da dimensão.
- Segundo parágrafo: apresente os principais desafios, contrastes e lacunas e
  interprete o que eles significam para a gestão municipal. Não apenas repita
  os indicadores já mencionados no primeiro parágrafo.
- Cada parágrafo deve ser desenvolvido o suficiente para formar uma análise,
  preferencialmente com 4 a 6 frases curtas ou médias.
- Quando houver resultados contrastantes, explique claramente a diferença.
- Evite transformar a análise em uma enumeração de indicadores.
- Os dois parágrafos devem ter funções diferentes e complementares.

SUGESTÕES DE MELHORIA:
- Produza exatamente 4 sugestões.
- Cada sugestão deve responder a uma lacuna ou necessidade identificada nos
  INDICADORES DA DIMENSÃO.
- As quatro sugestões devem ser distintas entre si e evitar reformulações da
  mesma recomendação.
- Priorize ações concretas e executáveis pela gestão municipal.
- Quando os indicadores permitirem, varie o tipo de ação entre gestão e
  planejamento, infraestrutura ou tecnologia, integração de processos/dados e
  melhoria do serviço oferecido à população.
- Não crie uma sugestão apenas para completar a quantidade. Toda sugestão deve
  ter justificativa clara nos indicadores fornecidos.
- Os DADOS CONTEXTUAIS podem ajudar a caracterizar a análise, mas não devem ser
  a única origem de uma sugestão.

USO DA PESQUISA WEB NA DIMENSÃO — REGRAS OBRIGATÓRIAS:
- Os INDICADORES DA PLANILHA continuam sendo a fonte principal do diagnóstico e
  dos níveis de maturidade. A pesquisa web NÃO recalcula e NÃO substitui esses dados.
- Use a pesquisa web apenas para VALIDAR, QUALIFICAR ou ATUALIZAR fatos relevantes
  diretamente relacionados aos indicadores da dimensão.
- Dê atenção especial a obras, programas, contratos, implantações, ampliações ou
  mudanças recentes que possam tornar uma leitura do indicador desatualizada ou
  incompleta.
- Se o indicador apontar uma deficiência, mas a pesquisa mostrar uma ação recente
  em andamento para enfrentá-la, mantenha a deficiência como situação medida e
  informe de forma breve que existe uma iniciativa em curso.
- Diferencie rigorosamente: ANUNCIADO/PLANEJADO, LICITADO/CONTRATADO,
  EM IMPLANTAÇÃO/EM CONSTRUÇÃO e CONCLUÍDO/EM OPERAÇÃO. Nunca trate obra ou projeto
  em andamento como resultado já alcançado.
- Se a informação externa apenas usar metodologia, período ou definição diferente
  da planilha, não declare que o indicador está errado.
- Quando houver divergência não conciliável, preserve a planilha como referência
  do diagnóstico e omita a informação externa do texto final.
- Use no máximo DOIS fatos externos em toda a análise da dimensão, apenas quando
  realmente mudarem ou qualificarem a interpretação.
- A pesquisa web também deve evitar recomendações ultrapassadas: se uma medida já
  estiver comprovadamente em execução, não recomende simplesmente "implantar" a
  mesma medida. Prefira, quando sustentado pelos indicadores, concluir, ampliar,
  integrar, monitorar ou avaliar a ação em andamento.
- Não crie uma sugestão baseada SOMENTE em uma notícia ou informação da internet;
  ela precisa continuar relacionada a uma lacuna ou necessidade dos indicadores.

REGRAS ABSOLUTAS:
- NÃO encerrar o texto antes de concluir TODAS as partes
- NÃO usar números, índices ou valores
- NÃO mencionar inteligência artificial
- NÃO mencionar a base conceitual ou documentos de referência
- Linguagem acessível, institucional e clara
- Evitar linguagem excessivamente técnica ou abstrata
- Priorizar frases mais diretas e naturais
- Escrever como um diagnóstico institucional real,
  e não como texto acadêmico ou promocional

Regras obrigatórias:

Utilize linguagem técnica, mas simples e direta.
Priorize frases de tamanho curto ou médio.
Descreva os resultados encontrados antes de fazer interpretações.
Evite elogios excessivos ao município.
Evite transformar todo resultado positivo em uma "potencialidade" ou todo resultado negativo em uma "oportunidade".
Não utilize linguagem de consultoria, marketing ou textos promocionais.
Evite conclusões genéricas que não estejam diretamente relacionadas aos indicadores analisados.
Não exagere a importância dos resultados.
Quando houver aspectos positivos e negativos, apresente-os de maneira equilibrada e factual.
As recomendações devem ser práticas e compatíveis com os problemas identificados.

EVITE EXPRESSÕES TÍPICAS DE TEXTOS GERADOS POR IA, como:

"pilares robustos"
"ambiente fértil"
"potencial significativo"
"caminho promissor"
"desafios significativos"
"oportunidades estratégicas"
"ativo valioso"
"efervescência"
"impulsionar a inovação"
"elevar a eficiência"
"fortalecer a visão de futuro"
"representa uma oportunidade"
"demonstra sólido compromisso"
"se destaca por"
"é fundamental para"
"desempenha papel fundamental"

Também evite iniciar repetidamente os parágrafos com construções como:

"O município demonstra..."
"O município apresenta..."
"Viçosa demonstra..."
"Destaca-se..."
"Observa-se que..."
"Percebe-se que..."

Varie a construção das frases de forma natural.

PREFIRA formulações mais concretas.

Em vez de:
"O município demonstra um sólido compromisso com a transformação digital."

Escreva:
"O município já oferece parte dos serviços públicos pela internet e utiliza sistemas digitais em algumas áreas da administração."

Em vez de:
"Esse cenário representa uma oportunidade estratégica para ampliar a inovação."

Escreva:
"Esses serviços ainda podem ser ampliados e integrados."

Em vez de:
"A presença das universidades cria um ambiente fértil para a inovação."

Escreva:
"A presença das universidades facilita a aproximação entre a prefeitura, pesquisadores e empresas locais."

Não invente benefícios, causas ou relações que não estejam sustentadas pelos indicadores fornecidos.
Quando usar informação externa, ela deve estar explicitamente presente na PESQUISA WEB DE VALIDAÇÃO
fornecida e deve ser apresentada com o grau correto de execução (planejada, contratada, em andamento
ou concluída), sem antecipar resultados.
"""

prompt_geral = ChatPromptTemplate.from_template(TEMPLATE_ANALISE_GERAL)
prompt_dimensao = ChatPromptTemplate.from_template(TEMPLATE_ANALISE_DIMENSAO)

chain_geral = prompt_geral | llm_estruturado_geral
chain_dimensao = prompt_dimensao | llm_estruturado_dimensao


In [ ]:
# =====================================================
# CHAMADAS AO MODELO (com validações e novas assinaturas)
# =====================================================
def _formatar_indicadores(indicadores: dict) -> str:
    return "\n".join(f"- {k}: {v}" for k, v in indicadores.items())


def _formatar_chunks(chunks: list) -> str:
    return "\n\n".join(f"[{c['topico']}] {c['texto']}" for c in chunks)


def _formatar_dados_contextuais(dados_contextuais: dict) -> str:
    """Formata os dados contextuais (cadastrais/socioeconômicos) para o
    prompt. Usados apenas para caracterizar o cenário — nunca tratados
    como indicadores nem como base de sugestões."""
    if not dados_contextuais:
        return "Nenhum dado contextual disponível para este recorte."
    return "\n".join(f"- {k}: {v}" for k, v in dados_contextuais.items())


def _extrair_texto_resposta(resposta) -> str:
    """Extrai texto de uma AIMessage, inclusive quando Gemini devolve
    blocos de conteúdo por causa do Google Search Grounding."""
    texto = getattr(resposta, "text", None)
    if isinstance(texto, str) and texto.strip():
        return texto.strip()

    conteudo = getattr(resposta, "content", "")
    if isinstance(conteudo, str):
        return conteudo.strip()

    partes = []
    if isinstance(conteudo, list):
        for bloco in conteudo:
            if isinstance(bloco, str):
                partes.append(bloco)
            elif isinstance(bloco, dict) and bloco.get("type") == "text":
                partes.append(str(bloco.get("text", "")))
    return "\n".join(p for p in partes if p.strip()).strip()


def _normalizar_codigo_ibge(codigo_ibge):
    """Normaliza o código IBGE vindo da planilha sem inventar valor."""
    if codigo_ibge is None or pd.isna(codigo_ibge):
        return None
    try:
        # Excel pode carregar o código como float (ex.: 3171303.0).
        if isinstance(codigo_ibge, (int, float)):
            return str(int(codigo_ibge))
    except Exception:
        pass
    texto = str(codigo_ibge).strip()
    if texto.endswith('.0') and texto[:-2].isdigit():
        texto = texto[:-2]
    return texto or None


def _identificacao_municipio(municipio, estado=None, codigo_ibge=None):
    codigo = _normalizar_codigo_ibge(codigo_ibge)
    uf_txt = str(estado).strip() if estado is not None and not pd.isna(estado) else "UF não informada"
    codigo_txt = codigo if codigo else "não informado"
    return uf_txt, codigo_txt


def pesquisar_contexto_municipio(municipio, estado=None, codigo_ibge=None, tentativas=2):
    """Faz pesquisa web curta para complementar SOMENTE a Análise Geral.

    A identidade territorial deve ser confirmada por nome + UF e, quando
    disponível, pelo código IBGE. Resultados de municípios homônimos são
    descartados. Em caso de dúvida, a função prefere não usar o fato.
    """
    uf_txt, codigo_txt = _identificacao_municipio(municipio, estado, codigo_ibge)
    localizacao = f"{municipio}, {uf_txt}, Brasil"

    prompt_pesquisa = f"""
Faça uma pesquisa BREVE na internet sobre o município abaixo para apoiar a
contextualização de um diagnóstico institucional municipal. Use no máximo
3 consultas de busca.

IDENTIFICAÇÃO TERRITORIAL OBRIGATÓRIA:
- Município: {municipio}
- UF: {uf_txt}
- Código IBGE: {codigo_txt}

Antes de utilizar QUALQUER fato, confirme que a fonte se refere especificamente
a esse município e a essa UF. Quando o código IBGE estiver disponível, use-o
como elemento adicional de conferência. Ignore resultados de municípios
homônimos ou de localidades com nome semelhante. Se a identidade territorial
não estiver clara na fonte, DESCARTE a informação.

Priorize fontes nesta ordem:
1. Prefeitura e outros órgãos públicos;
2. IBGE e órgãos oficiais estaduais/federais;
3. universidades e instituições públicas de ensino/pesquisa;
4. imprensa confiável, apenas quando realmente necessário.

Procure somente fatos úteis e relativamente estáveis sobre:
- perfil econômico e territorial;
- presença de universidades, centros de pesquisa ou ecossistema de inovação;
- papel regional do município;
- infraestrutura ou serviços públicos relevantes;
- iniciativas públicas recentes que ajudem a compreender o contexto local.

Regras:
- NÃO faça avaliação de maturidade do município;
- NÃO dê recomendações;
- NÃO infira relações de causa e efeito;
- NÃO use textos promocionais, rankings comerciais ou opiniões como fatos;
- prefira fatos confirmados por fontes institucionais;
- seja seletivo: traga apenas 4 a 6 pontos curtos;
- em cada ponto, indique entre parênteses a fonte ou instituição de origem;
- se houver dúvida territorial, omita o ponto em vez de arriscar atribuí-lo
  ao município errado.

Se não encontrar informações confiáveis e úteis, responda exatamente:
"Pesquisa web sem contexto adicional confiável."
"""

    ultimo_erro = None
    for _ in range(tentativas):
        try:
            resposta = llm_pesquisa_web.invoke(prompt_pesquisa)
            texto = _extrair_texto_resposta(resposta)
            if texto:
                return texto
        except Exception as e:
            ultimo_erro = e
            print(f"⚠️  Falha na pesquisa web sobre {municipio}, tentando novamente... ({e})")

    if ultimo_erro:
        print(f"⚠️  Pesquisa web indisponível. A Análise Geral seguirá sem contexto externo. ({ultimo_erro})")
    return "Pesquisa web sem contexto adicional confiável."


def pesquisar_validacao_dimensao(
    dimensao,
    municipio,
    indicadores_dimensao,
    estado=None,
    codigo_ibge=None,
    tentativas=2,
):
    """Pesquisa web breve para validar/atualizar pontos de UMA dimensão.

    Não substitui os indicadores. Procura principalmente fatos recentes que
    qualifiquem a leitura do diagnóstico, como obras, programas, ampliações ou
    implantações em andamento. Se nada relevante for encontrado, retorna um
    marcador neutro e a análise segue somente com os dados da planilha.
    """
    uf_txt, codigo_txt = _identificacao_municipio(municipio, estado, codigo_ibge)
    indicadores_txt = _formatar_indicadores(indicadores_dimensao)

    prompt_pesquisa = f"""
Faça uma pesquisa BREVE na internet para VALIDAR E ATUALIZAR informações
relevantes da dimensão "{dimensao}" de um diagnóstico municipal. Use no máximo
3 consultas de busca e seja muito seletivo.

IDENTIFICAÇÃO TERRITORIAL OBRIGATÓRIA:
- Município: {municipio}
- UF: {uf_txt}
- Código IBGE: {codigo_txt}

INDICADORES DA PLANILHA QUE DEVEM ORIENTAR A PESQUISA:
{indicadores_txt}

OBJETIVO DA PESQUISA:
Localizar somente informações externas que ajudem a confirmar, qualificar ou
atualizar os pontos mais importantes desses indicadores. Dê prioridade a ações
recentes que possam mudar a interpretação prática do diagnóstico, como:
- obra pública em andamento;
- novo equipamento ou infraestrutura em implantação;
- programa, serviço ou sistema recentemente lançado;
- contrato, licitação ou expansão comprovada;
- alteração institucional relevante já formalizada.

Exemplo de interpretação correta: se a planilha aponta baixo tratamento de
esgoto, mas existe uma nova estação de tratamento em construção, NÃO diga que
o problema já foi resolvido. Registre que o indicador mostra uma deficiência
atual/histórica e que há uma ação em andamento que pode alterar esse cenário
quando concluída e operacionalizada.

VALIDAÇÃO TERRITORIAL:
Antes de usar qualquer fato, confirme que ele pertence especificamente a
{municipio}/{uf_txt}. Quando disponível, confira também o código IBGE
{codigo_txt}. Descarte resultados de municípios homônimos, órgãos de outra
localidade ou fontes cuja localização não seja clara.

FONTES — prioridade:
1. Prefeitura, autarquias e prestadores públicos locais;
2. Governo estadual/federal, IBGE, SNIS/SINISA, ministérios e agências oficiais;
3. universidades e instituições públicas de pesquisa;
4. imprensa confiável apenas como complemento, preferencialmente quando citar
   documento, obra, contrato ou autoridade identificável.

REGRAS DE CONFIABILIDADE E ATUALIDADE:
- Priorize informações recentes e páginas com data identificável.
- Diferencie claramente: anunciado/planejado; licitado/contratado;
  em implantação/em construção; concluído/em operação.
- Não trate anúncio político ou intenção como execução comprovada.
- Não use ranking comercial, texto promocional, postagem sem fonte ou opinião.
- Não declare que a planilha está errada apenas porque a web traz número
  diferente; datas, métodos e definições podem ser distintos.
- A web NÃO deve recalcular o nível de maturidade.
- Traga no máximo 3 achados realmente relevantes para esta dimensão.
- Para cada achado, informe: tema; fato atual; estágio da ação; fonte/instituição
  e data/ano quando disponível.
- Se não houver informação externa que realmente melhore a leitura da dimensão,
  não force resultados.

Se não encontrar informação confiável, diretamente relacionada aos indicadores
ou territorialmente segura, responda exatamente:
"Pesquisa web sem atualização relevante para esta dimensão."
"""

    ultimo_erro = None
    for _ in range(tentativas):
        try:
            resposta = llm_pesquisa_web.invoke(prompt_pesquisa)
            texto = _extrair_texto_resposta(resposta)
            if texto:
                return texto
        except Exception as e:
            ultimo_erro = e
            print(
                f"⚠️  Falha na validação web da dimensão '{dimensao}', "
                f"tentando novamente... ({e})"
            )

    if ultimo_erro:
        print(
            f"⚠️  Validação web indisponível para '{dimensao}'. "
            f"A análise seguirá com a planilha. ({ultimo_erro})"
        )
    return "Pesquisa web sem atualização relevante para esta dimensão."


def gerar_analise_geral(municipio, porte, indicadores, chunks_gerais, dados_contextuais=None, pesquisa_web=None, tentativas=2):
    """Gera a análise institucional geral (2 parágrafos).

    dados_contextuais: dados cadastrais/socioeconômicos (todas as
    dimensões) usados apenas para caracterizar o cenário do município —
    nunca tratados como indicadores nem usados para gerar números.

    pesquisa_web: síntese factual e breve recuperada na internet, usada
    apenas como contexto complementar da Análise Geral.
    """
    if chunks_gerais:
        instrucao_base = INSTRUCAO_COM_BASE.format(base_conceitual=_formatar_chunks(chunks_gerais))
    else:
        instrucao_base = INSTRUCAO_SEM_BASE
    indicadores_txt = _formatar_indicadores(indicadores)
    dados_contextuais_txt = _formatar_dados_contextuais(dados_contextuais or {})
    pesquisa_web_txt = pesquisa_web or "Pesquisa web sem contexto adicional confiável."

    ultimo_erro = None
    for _ in range(tentativas):
        try:
            resultado = chain_geral.invoke({
                "municipio": municipio, "porte": porte,
                "indicadores_txt": indicadores_txt, "instrucao_base": instrucao_base,
                "dados_contextuais_txt": dados_contextuais_txt,
                "pesquisa_web_txt": pesquisa_web_txt,
            })
            if not resultado.analise_geral or len(resultado.analise_geral.strip()) < 40:
                raise ValueError("Resposta do modelo vazia ou incompleta.")
            return resultado.analise_geral
        except Exception as e:  # resposta inválida do modelo
            ultimo_erro = e
            print(f"⚠️  Falha ao gerar análise geral, tentando novamente... ({e})")

    raise RuntimeError(f"Não foi possível gerar a análise geral após {tentativas} tentativas: {ultimo_erro}")



def _separar_paragrafos(texto: str) -> list:
    """Separa parágrafos reais, tolerando espaços nas linhas em branco."""
    return [p.strip() for p in re.split(r"\n\s*\n", texto.strip()) if p.strip()]

def gerar_analise_dimensao(
    dimensao,
    municipio,
    porte,
    indicadores_dimensao,
    dados_contextuais_dimensao=None,
    pesquisa_web_dimensao=None,
    linha_planilha=None,
    tentativas=2,
):
    """Gera análise em 2 parágrafos + exatamente 4 sugestões para uma dimensão.

    dados_contextuais_dimensao: dados cadastrais/socioeconômicos e
    institucionais associados a esta dimensão (via DADOS_CONTEXTUAIS_MAPA).

    pesquisa_web_dimensao: checagem externa breve e recente, usada somente
    para validar/qualificar a leitura dos indicadores e evitar recomendações
    desatualizadas. Nunca recalcula nível de maturidade nem substitui a planilha.
    """
    if not indicadores_dimensao:
        raise ValueError(f"A dimensão '{dimensao}' não possui indicadores associados.")

    chunks_selecionados = selecionar_chunks_dimensao(dimensao, indicadores_dimensao, linha_planilha)
    # separa os chunks gerais (sempre presentes) dos chunks setoriais, só
    # para decidir se existe base setorial específica para esta dimensão
    chunks_setoriais = [c for c in chunks_selecionados if c["dimensao"] != "Geral"]
    if chunks_selecionados:
        instrucao_base = INSTRUCAO_COM_BASE.format(base_conceitual=_formatar_chunks(chunks_selecionados))
    else:
        instrucao_base = INSTRUCAO_SEM_BASE
    if not chunks_setoriais:
        print(f"⚠️  Dimensão '{dimensao}': nenhum chunk setorial específico na Carta "
              f"(usando apenas chunks gerais e os indicadores como base).")

    indicadores_txt = _formatar_indicadores(indicadores_dimensao)
    dados_contextuais_txt = _formatar_dados_contextuais(dados_contextuais_dimensao or {})
    pesquisa_web_dimensao_txt = (
        pesquisa_web_dimensao
        or "Pesquisa web sem atualização relevante para esta dimensão."
    )

    ultimo_erro = None
    for _ in range(tentativas):
        try:
            resultado = chain_dimensao.invoke({
                "municipio": municipio, "porte": porte, "dimensao": dimensao,
                "indicadores_txt": indicadores_txt, "instrucao_base": instrucao_base,
                "dados_contextuais_txt": dados_contextuais_txt,
                "pesquisa_web_dimensao_txt": pesquisa_web_dimensao_txt,
            })
            if len(resultado.sugestoes) != 4:
                raise ValueError(f"Modelo retornou {len(resultado.sugestoes)} sugestões (esperado: 4).")
            if not resultado.analise or len(resultado.analise.strip()) < 80:
                raise ValueError("Análise da dimensão vazia ou curta demais.")
            paragrafos = _separar_paragrafos(resultado.analise)
            if len(paragrafos) != 2:
                raise ValueError(
                    f"Modelo retornou {len(paragrafos)} parágrafo(s) na análise (esperado: 2)."
                )
            analise_formatada = "\n\n".join(paragrafos)
            return {"analise": analise_formatada, "sugestoes": resultado.sugestoes}
        except Exception as e:
            ultimo_erro = e
            print(f"⚠️  Falha ao gerar análise de '{dimensao}', tentando novamente... ({e})")

    raise RuntimeError(f"Não foi possível gerar a análise de '{dimensao}' após {tentativas} tentativas: {ultimo_erro}")


In [ ]:
# =====================================================
# GERAÇÃO DO RELATÓRIO WORD (adaptado à saída estruturada)
# =====================================================
def gerar_relatorio_word(municipio, relatorio: dict):
    doc = Document()
    doc.add_heading(f"Relatório Institucional – {municipio}", level=1)

    doc.add_heading("Análise Geral", level=2)
    for paragrafo in _separar_paragrafos(relatorio["analise_geral"]):
        doc.add_paragraph(paragrafo)

    nomes_exibicao = {
        "economica": "Dimensão Econômica",
        "sociocultural": "Dimensão Sociocultural",
        "meio_ambiente": "Dimensão Meio Ambiente",
        "capacidades_institucionais": "Dimensão Capacidades Institucionais",
    }

    for chave, titulo in nomes_exibicao.items():
        bloco = relatorio["dimensoes"][chave]
        doc.add_heading(titulo, level=2)
        for paragrafo in _separar_paragrafos(bloco["analise"]):
            doc.add_paragraph(paragrafo)
        doc.add_paragraph("Sugestões de melhoria:")
        for sugestao in bloco["sugestoes"]:
            doc.add_paragraph(sugestao, style="List Bullet")

    nome_arquivo = f"{BASE_PATH}/Relatorio_{municipio.replace(' ', '_')}.docx"
    doc.save(nome_arquivo)
    print(f"📄 Relatório gerado: {nome_arquivo}")


In [ ]:
# =====================================================
# EXECUÇÃO PRINCIPAL
# =====================================================
DIMENSAO_PARA_CHAVE = {
    "Econômica": "economica",
    "Sociocultural": "sociocultural",
    "Meio Ambiente": "meio_ambiente",
    "Capacidades Institucionais": "capacidades_institucionais",
}


def main():
    # --- Planilha ---
    try:
        df = pd.read_excel(ARQUIVO_PLANILHA)
    except FileNotFoundError:
        raise FileNotFoundError(f"Planilha de indicadores não encontrada: {ARQUIVO_PLANILHA}")

    df.columns = (
        df.columns.str.strip().str.lower().str.normalize("NFKD")
        .str.encode("ascii", errors="ignore").str.decode("utf-8")
    )

    municipios = df[COLUNA_MUNICIPIO].unique().tolist()
    print("\nMunicípios disponíveis:")
    for i, m in enumerate(municipios, 1):
        print(f"{i} - {m}")

    escolha = int(input("\nDigite o número do município: "))
    municipio = municipios[escolha - 1]

    dados = df[df[COLUNA_MUNICIPIO] == municipio].iloc[0]
    populacao = int(dados[COLUNA_POPULACAO])
    porte = classificar_porte(populacao)
    estado = dados.get("estado", None)
    codigo_ibge = _normalizar_codigo_ibge(dados.get("cod municipio", None))
    print(f"\n🔹 Porte identificado: {porte}")
    if codigo_ibge:
        print(f"🔹 Código IBGE usado para validação territorial: {codigo_ibge}")

    colunas_ignoradas = [COLUNA_MUNICIPIO, COLUNA_POPULACAO, "cod municipio", "estado", "avaliada"]
    candidatos_indicadores = dados.drop(
        labels=[c for c in colunas_ignoradas if c in dados.index]
    ).to_dict()

    # --- Classifica cada coluna candidata por dimensão/tópico ---
    # (colunas de metadado/agregado em COLUNAS_NAO_INDICADORES são
    # descartadas aqui mesmo, sem entrar em INDICADORES_NAO_CLASSIFICADOS)
    indicadores_por_dimensao = {d: {} for d in DIMENSOES}
    for nome, valor in candidatos_indicadores.items():
        classificacao = classificar_indicador(nome)
        if classificacao:
            indicadores_por_dimensao[classificacao["dimensao"]][nome] = valor

    # análise geral usa o conjunto de todos os indicadores já classificados
    # (não os brutos da planilha, para não incluir ruído/agregados)
    todos_indicadores = {
        nome: valor
        for indicadores in indicadores_por_dimensao.values()
        for nome, valor in indicadores.items()
    }

    # --- Dados contextuais (cadastrais/socioeconômicos e institucionais) ---
    # Usados apenas para complementar a Análise Geral e a análise da
    # dimensão correspondente — nunca como indicador, nunca para nível de
    # maturidade/priorização e nunca como origem direta de sugestões.
    dados_contextuais_por_dimensao = {
        dimensao: obter_dados_contextuais(dimensao, dados) for dimensao in DIMENSOES
    }
    todos_dados_contextuais = obter_todos_dados_contextuais(dados)

    for dimensao in DIMENSOES:
        if not indicadores_por_dimensao[dimensao]:
            print(f"⚠️  Dimensão '{dimensao}' ficou sem indicadores classificados.")

    if INDICADORES_NAO_CLASSIFICADOS:
        print(f"⚠️  Indicadores não classificados (revisar MAPA_INDICADORES): "
              f"{sorted(set(INDICADORES_NAO_CLASSIFICADOS))}")

    # --- Até 10 chamadas ao modelo:
    # 1 pesquisa web geral + 1 análise geral +
    # 4 pesquisas web de validação (uma por dimensão) + 4 análises dimensionais.
    # Se uma pesquisa web falhar, o relatório continua usando a planilha. ---
    chunks_gerais = [CHUNKS_CARTA[f"geral_{t}"] for t in CHUNKS_GERAIS if f"geral_{t}" in CHUNKS_CARTA]

    print("🔎 Fazendo pesquisa web breve sobre o município...")
    pesquisa_web = pesquisar_contexto_municipio(
        municipio,
        estado=estado,
        codigo_ibge=codigo_ibge,
    )
    print("🔹 Contexto web recuperado:")
    print(pesquisa_web)

    print("🔹 Gerando análise geral...")
    analise_geral = gerar_analise_geral(
        municipio, porte, todos_indicadores, chunks_gerais,
        dados_contextuais=todos_dados_contextuais,
        pesquisa_web=pesquisa_web,
    )

    relatorio = {"analise_geral": analise_geral, "dimensoes": {}}
    for dimensao, chave in DIMENSAO_PARA_CHAVE.items():
        print(f"🔎 Validando na web os principais pontos da dimensão {dimensao}...")
        pesquisa_web_dimensao = pesquisar_validacao_dimensao(
            dimensao=dimensao,
            municipio=municipio,
            indicadores_dimensao=indicadores_por_dimensao[dimensao],
            estado=estado,
            codigo_ibge=codigo_ibge,
        )
        print(f"🔹 Validação web — {dimensao}:")
        print(pesquisa_web_dimensao)

        print(f"🔹 Gerando análise da dimensão {dimensao}...")
        relatorio["dimensoes"][chave] = gerar_analise_dimensao(
            dimensao=dimensao,
            municipio=municipio,
            porte=porte,
            indicadores_dimensao=indicadores_por_dimensao[dimensao],
            dados_contextuais_dimensao=dados_contextuais_por_dimensao[dimensao],
            pesquisa_web_dimensao=pesquisa_web_dimensao,
            linha_planilha=dados,
        )

    print("\n" + json.dumps(relatorio, ensure_ascii=False, indent=2))

    gerar_relatorio_word(municipio, relatorio)
    print("✅ Processo finalizado com sucesso.")


main()
